# 🌊 DAQathon Session 1: QC with Random Forests, k-means, CNNs, and Transformers

This notebook is written to meet you where you are. You might already be comfortable with machine learning, or you might be seeing terms like *Random Forest*, *k-means*, *convolutional neural network*, or *transformer* for the first time.

The goal is to make the workflow feel approachable while still using a real ONC quality-control problem:

- start from raw scalar ONC data and target labels,
- build one clear supervised baseline with a Random Forest,
- build one clear unsupervised example with k-means,
- compare two sequence-model baselines with a CNN and a small transformer,
- and leave with a better sense of how to improve the pipeline later.


## 🧰 One-Time FIR Setup

Before you use these notebooks on FIR for the first time, open a FIR terminal and run:

```bash
jupyter kernelspec install --user /project/def-kmoran/shared/daqathon/kernels/daqathon-ml
```

This is a **one-time per-user step**. After that:

- refresh JupyterHub if it was already open,
- open the notebook from your cloned repo,
- select the shared `Daqathon ML` kernel.


## 🧭 How To Use This Notebook

This notebook is meant to feel like a first clean pass through the workflow:

- understand the dataset and target labels,
- build one reasonable tabular baseline with a Random Forest,
- explore one unsupervised baseline with k-means,
- build two sequential baselines with a CNN and a transformer,
- then ask what we might improve next.

A few suggestions while you work through the notebook:

- read the markdown cells first, then run the code cells underneath them,
- use `DATA_FRACTION` in the config cell to switch between a quick demo and a longer run,
- run the CNN and transformer sections when you want to compare sequence models,
- treat the default model settings as **baseline choices**, not as final answers.

Optional background reading:

- [Random Forests in scikit-learn](https://scikit-learn.org/stable/modules/ensemble.html#forest)
- [k-means clustering in scikit-learn](https://scikit-learn.org/stable/modules/clustering.html#k-means)
- [Neural networks in PyTorch](https://docs.pytorch.org/tutorials/beginner/blitz/neural_networks_tutorial)
- [The Illustrated Transformer](https://jalammar.github.io/illustrated-transformer/)
- [ONC QAQC reference](https://wiki.oceannetworks.ca/spaces/DP/pages/42174414/Quality+Assurance+Quality+Control)


## ⚙️ Configuration

The next few cells separate the main configuration ideas:

- choose a dataset preset,
- optionally override a few important dataset-specific fields,
- then adjust broad run controls like how much data to use.

On FIR, if `$SLURM_TMPDIR` is available, this notebook first stages the shared raw CSV directory and the prepared parquet cache into node-local job storage. It writes all generated outputs into a runtime directory chosen in this order:

1. `$SLURM_TMPDIR`
2. `$SCRATCH`
3. repo-local `tmp/session1_outputs`


In [ ]:
from pathlib import Path
import sys

# Find the repo root before importing utilities from scripts/.
NOTEBOOK_ROOT = Path.cwd()
for candidate_root in [NOTEBOOK_ROOT, *NOTEBOOK_ROOT.parents]:
    if (candidate_root / "notebooks").exists() and (candidate_root / "scripts").exists():
        NOTEBOOK_ROOT = candidate_root
        break

if str(NOTEBOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_ROOT))

from scripts.session1_notebook_bootstrap import build_notebook_bootstrap_namespace

globals().update(build_notebook_bootstrap_namespace(NOTEBOOK_ROOT))
show_setup_json(
    {
        "NOTEBOOK_ROOT": NOTEBOOK_ROOT,
        "SHARED_DAQATHON_ROOT": SHARED_DAQATHON_ROOT,
        "LOCAL_CACHE_DIR": LOCAL_CACHE_DIR,
        "SHARED_CACHE_DIR": SHARED_CACHE_DIR,
    }
)


### Dataset Presets

The next cell defines the **supported scalar dataset presets** for this notebook.

Each preset bundles together the choices that usually travel as a group:

- where the raw files live,
- which label column is the supervised target,
- which measurement columns to use,
- which two columns should appear in plots,
- which device family is the primary stream,
- and whether k-means should work on window summaries or row-level features.

In most cases, you only need to change `DATASET_PROFILE_ID`.


In [ ]:
# Each profile captures the dataset-specific choices that would otherwise be
# repeated throughout the notebook. Once a profile is selected, later cells
# work with normalised variables such as TARGET_FLAG or MEASUREMENT_COLUMNS.
DATASET_PROFILES = {
    "ctd_conductivity": {
        # A classic CTD example: one instrument export with conductivity QC as the label.
        "label": "CTD conductivity QC",
        "description": "Strait of Georgia East CTD data with conductivity QC as the supervised target.",
        "raw_subpaths": ["SoGEast_CTD_202503_202603"],
        "cache_stem": "conductivity_scalar_session1",
        "target_flag": "Conductivity QC Flag",
        "task_mode": "multiclass",
        "good_labels": [1],
        "issue_labels": [3, 4, 9],
        "flag_example_classes": [1, 3, 4, 9],
        "measurement_columns": [
            "Conductivity (S/m)",
            "Density (kg/m3)",
            "Depth (m)",
            "Practical Salinity (psu)",
            "Pressure (decibar)",
            "Sigma-t (kg/m3)",
            "Sigma-theta (0 dbar) (kg/m3)",
            "Sound Speed (m/s)",
            "Temperature (C)",
        ],
        "optional_qc_columns": ["Temperature QC Flag"],
        "plot_measurement_column": "Conductivity (S/m)",
        "plot_secondary_column": "Temperature (C)",
        "primary_device": "ctd",
        "kmeans_feature_mode": "window_summary",
        "default_sequence_output_mode": "window",
        "default_sequence_target_strategy": "collapsed_1_34_9",
        "cache_read_sample_rows": None,
        "auto_build_missing_cache": True,
        "default_prep_sample_rows": None,
    },
    "fluorometer_turbidity": {
        # A merged scalar example: fluorometer is the primary device, but CTD and oxygen
        # provide useful context columns during the merge.
        "label": "Fluorometer turbidity QC",
        "description": "Merged scalar data around a fluorometer/turbidity target, including CTD and oxygen context columns.",
        "raw_subpaths": ["Fluorometer/SoGCentral", "Fluorometer/Folger", "Fluorometer/SoGCentral_test", "SoGCentral_test"],
        "cache_stem": "sogcentral_turbidity",
        "target_flag": "Turbidity QC Flag",
        "task_mode": "multiclass",
        "good_labels": [1],
        "issue_labels": [3, 4, 9],
        "flag_example_classes": [1, 3, 4, 9],
        "measurement_columns": [
            "Chlorophyll (ug/l)",
            "Turbidity (NTU)",
            "Conductivity (S/m)",
            "Density (kg/m3)",
            "Practical Salinity (psu)",
            "Pressure (decibar)",
            "Sigma-t (kg/m3)",
            "Sigma-theta (0 dbar) (kg/m3)",
            "Sound Speed (m/s)",
            "Temperature (C)",
            "Oxygen Concentration Corrected (ml/l)",
            "Oxygen Concentration Uncorrected (ml/l)",
        ],
        "optional_qc_columns": ["Chlorophyll QC Flag"],
        "plot_measurement_column": "Turbidity (NTU)",
        "plot_secondary_column": "Temperature (C)",
        "primary_device": "fluorometer",
        "kmeans_feature_mode": "row_level",
        "default_sequence_output_mode": "per_timestep",
        "default_sequence_target_strategy": "collapsed_1_34_9",
        "cache_read_sample_rows": None,
        "auto_build_missing_cache": True,
        "default_prep_sample_rows": None,
    },
    "oxygen": {
        # A smaller oxygen-focused profile for the same scalar workflow.
        "label": "Oxygen QC",
        "description": "Scalar oxygen data with oxygen concentration QC as the supervised target.",
        "raw_subpaths": ["Fluorometer/SoGCentral", "Fluorometer/Folger", "SoGEast_Oxygen_202503_202603", "Oxygen", "oxygen"],
        "cache_stem": "sogcentral_oxygen",
        "target_flag": "Oxygen Concentration Corrected QC Flag",
        "task_mode": "multiclass",
        "good_labels": [1],
        "issue_labels": [3, 4, 9],
        "flag_example_classes": [1, 2, 3, 4, 9],
        "measurement_columns": [
            "Oxygen Concentration Corrected (ml/l)",
            "Oxygen Concentration Uncorrected (ml/l)",
            "Temperature (C)",
            "Pressure (decibar)",
        ],
        "optional_qc_columns": ["Temperature QC Flag"],
        "plot_measurement_column": "Oxygen Concentration Corrected (ml/l)",
        "plot_secondary_column": "Temperature (C)",
        "primary_device": "oxygen",
        "kmeans_feature_mode": "window_summary",
        "default_sequence_output_mode": "window",
        "default_sequence_target_strategy": "collapsed_12_34_9",
        "cache_read_sample_rows": None,
        "auto_build_missing_cache": True,
        "default_prep_sample_rows": None,
    },
    "conductivity_plugs": {
        # A labelled conductivity-plug example with a compact CSV schema.
        "label": "Conductivity plugs",
        "description": "Conductivity-plug data with a custom ml_label target and CTD/oxygen context columns.",
        "raw_subpaths": ["ConductivityPlugs"],
        "cache_stem": "conductivity_plugs_session1",
        "target_flag": "ml_label",
        "task_mode": "multiclass",
        # Custom labels for the conductivity-plug dataset:
        # 0 = good, 1 = conductivity bad plug, 2 = conductivity bad other,
        # 3 = non-conductivity failure, 4 = missing data.
        "good_labels": [0],
        "issue_labels": [1, 2, 3, 4],
        "flag_example_classes": [1, 2, 3, 4],
        "measurement_columns": [
            "cond_value_ctd",
            "density_value_ctd",
            "Pressure_value_ctd",
            "salinity_value_ctd",
            "sigmaT_value_ctd",
            "SIGMA_THETA_value_ctd",
            "Sound_Speed_value_ctd",
            "Temperature_value_ctd",
            "oxygen_corrected_value_oxy",
            "oxygen_uncorrected_value_oxy",
            "temperature_value_oxy",
            "temperature_offset",
            "temperature_offset_anomaly",
            "temperature_offset_over_start_mean",
        ],
        "optional_qc_columns": ["cond_qaqc_ctd", "oxygen_corrected_qaqc_oxy", "temperature_qaqc_oxy"],
        "plot_measurement_column": "cond_value_ctd",
        "plot_secondary_column": "Temperature_value_ctd",
        "primary_device": "other",
        "kmeans_feature_mode": "window_summary",
        "default_sequence_output_mode": "per_timestep",
        "default_sequence_target_strategy": "raw_multiclass",
        "cache_read_sample_rows": 250_000,
        "auto_build_missing_cache": False,
        "default_prep_sample_rows": 500_000,
    },
}

# Change this to switch the whole notebook to a different supported preset.
DATASET_PROFILE_ID = "ctd_conductivity"
DATASET_PROFILE = DATASET_PROFILES[DATASET_PROFILE_ID]

show_setup_json(
    {
        "DATASET_PROFILE_ID": DATASET_PROFILE_ID,
        "DATASET_LABEL": DATASET_PROFILE["label"],
        "DESCRIPTION": DATASET_PROFILE["description"],
        "PRIMARY_DEVICE": DATASET_PROFILE["primary_device"],
        "GOOD_LABELS": DATASET_PROFILE.get("good_labels", [1]),
        "ISSUE_LABELS": DATASET_PROFILE.get("issue_labels", [3, 4, 9]),
    }
)


### Optional Dataset Overrides

Presets are the normal starting point, but these overrides let you change a few high-value pieces without editing the rest of the notebook.

Leave them as `None` to use the preset values. Typical reasons to override something here are:

- testing a different raw folder,
- changing the target flag,
- narrowing the measurement columns,
- or forcing a different k-means feature mode.


In [ ]:
# Leave these as None unless you intentionally want to override part of the preset.
# The simplest workflow is "pick a preset, then tweak only one or two fields."
MANUAL_RAW_DATA_DIR = None
MANUAL_TARGET_FLAG = None
MANUAL_MEASUREMENT_COLUMNS = None
MANUAL_OPTIONAL_QC_COLUMNS = None
MANUAL_PLOT_MEASUREMENT_COLUMN = None
MANUAL_PLOT_SECONDARY_COLUMN = None
MANUAL_CACHE_BUNDLE_NAME = None
MANUAL_KMEANS_FEATURE_MODE = None

# Build a short list of likely raw-data locations for this profile.
# On FIR we prefer the shared project directory; locally we fall back to repo data.
profile_raw_candidates = [Path(subpath) for subpath in DATASET_PROFILE["raw_subpaths"]]
local_raw_candidates = [NOTEBOOK_ROOT / "data" / "raw" / subpath for subpath in profile_raw_candidates]
shared_raw_candidates = (
    [SHARED_DAQATHON_ROOT / "data" / "raw" / subpath for subpath in profile_raw_candidates]
    if SHARED_DAQATHON_ROOT
    else []
)

# Resolve the best available raw-data and cache locations without forcing
# you to edit paths in multiple notebook sections.
detected_raw_data_dir = (
    first_existing_csv_dir([candidate for candidate in [*shared_raw_candidates, *local_raw_candidates] if candidate is not None])
    or first_existing_path([candidate for candidate in [*shared_raw_candidates, *local_raw_candidates, NOTEBOOK_ROOT / "data" / "raw"] if candidate is not None])
)
detected_cache_dir = first_existing_path([candidate for candidate in [LOCAL_CACHE_DIR, SHARED_CACHE_DIR] if candidate is not None])

# Normalise everything into the small set of variables used by the rest of the notebook.
# From this point onward, later cells should not need to care which preset was chosen.
READ_RAW_DATA_DIR = MANUAL_RAW_DATA_DIR or (
    str(detected_raw_data_dir)
    if detected_raw_data_dir is not None
    else str(local_raw_candidates[0] if local_raw_candidates else NOTEBOOK_ROOT / "data" / "raw")
)
READ_CACHE_DIR = str(detected_cache_dir) if detected_cache_dir is not None else str(LOCAL_CACHE_DIR)

TARGET_FLAG = MANUAL_TARGET_FLAG or DATASET_PROFILE["target_flag"]
TASK_MODE = DATASET_PROFILE["task_mode"]
GOOD_LABELS = [int(label) for label in DATASET_PROFILE.get("good_labels", [1])]
ISSUE_LABELS = [int(label) for label in DATASET_PROFILE.get("issue_labels", [3, 4, 9])]
FLAG_EXAMPLE_CLASSES = tuple(int(label) for label in DATASET_PROFILE.get("flag_example_classes", [1, 3, 4, 9]))
CACHE_BUNDLE_NAME = MANUAL_CACHE_BUNDLE_NAME or DATASET_PROFILE["cache_stem"]
PRIMARY_DEVICE = DATASET_PROFILE["primary_device"]
KMEANS_FEATURE_MODE = MANUAL_KMEANS_FEATURE_MODE or DATASET_PROFILE["kmeans_feature_mode"]
DEFAULT_SEQUENCE_OUTPUT_MODE = DATASET_PROFILE.get("default_sequence_output_mode", "window")
DEFAULT_SEQUENCE_TARGET_STRATEGY = DATASET_PROFILE.get("default_sequence_target_strategy", "collapsed_1_34_9")

# Keep the feature list in one place so every later section reuses the same columns.
MEASUREMENT_COLUMNS = list(MANUAL_MEASUREMENT_COLUMNS or DATASET_PROFILE["measurement_columns"])
OPTIONAL_QC_COLUMNS = list(dict.fromkeys(MANUAL_OPTIONAL_QC_COLUMNS or DATASET_PROFILE["optional_qc_columns"]))
PLOT_MEASUREMENT_COLUMN = MANUAL_PLOT_MEASUREMENT_COLUMN or DATASET_PROFILE["plot_measurement_column"]
PLOT_SECONDARY_COLUMN = MANUAL_PLOT_SECONDARY_COLUMN or DATASET_PROFILE["plot_secondary_column"]

# These explicit column lists are one of the main reasons the notebook can stay fast:
# we only read the row-level or window-level fields we actually need downstream.
ROW_USE_COLUMNS = list(dict.fromkeys(["Time UTC", "source_file", TARGET_FLAG, *OPTIONAL_QC_COLUMNS, *MEASUREMENT_COLUMNS]))
WINDOW_FEATURE_COLUMNS = [
    f"{column}_{stat}"
    for column in MEASUREMENT_COLUMNS
    for stat in ("mean", "std")
]
WINDOW_USE_COLUMNS = [
    "window_start",
    "window_end",
    "source_file",
    "issue_rate",
    *WINDOW_FEATURE_COLUMNS,
]

show_setup_json(
    {
        "READ_RAW_DATA_DIR": READ_RAW_DATA_DIR,
        "READ_CACHE_DIR": READ_CACHE_DIR,
        "TARGET_FLAG": TARGET_FLAG,
        "CACHE_BUNDLE_NAME": CACHE_BUNDLE_NAME,
        "KMEANS_FEATURE_MODE": KMEANS_FEATURE_MODE,
        "DEFAULT_SEQUENCE_OUTPUT_MODE": DEFAULT_SEQUENCE_OUTPUT_MODE,
        "DEFAULT_SEQUENCE_TARGET_STRATEGY": DEFAULT_SEQUENCE_TARGET_STRATEGY,
        "GOOD_LABELS": GOOD_LABELS,
        "ISSUE_LABELS": ISSUE_LABELS,
        "MEASUREMENT_COLUMNS": MEASUREMENT_COLUMNS,
        "PLOT_MEASUREMENT_COLUMN": PLOT_MEASUREMENT_COLUMN,
        "PLOT_SECONDARY_COLUMN": PLOT_SECONDARY_COLUMN,
    }
)


## Run Controls

These are the notebook-wide knobs you are most likely to change.

The most important is `DATA_FRACTION`:

- use a small value for quick live demos,
- increase it when you want a stronger run,
- keep it near `1.0` when you want to use the largest training budget.

This cell also keeps the early raw-data preview explicit: you can choose how many files to inspect in Part 1 without affecting the fixed split choices later on.


In [ ]:
# One easy top-level knob: start at 0.1 for a quick run, then raise it if you want a bigger experiment.
DATA_FRACTION = 0.1
if not 0 < DATA_FRACTION <= 1:
    raise ValueError("DATA_FRACTION must be in the interval (0, 1].")

# Part 1 uses only a tiny raw-data preview so we can look at real examples quickly.
ORIENTATION_FILE_SELECTION_MODE = "spread"
ORIENTATION_FILE_LIMIT = 3
BASE_FLAG_EXAMPLE_POINTS_PER_PANEL = 30_000

# Split fractions are reused across the notebook.
TRAIN_FRACTION = 0.70
VALIDATION_FRACTION = 0.15

# Keep stochastic model training reproducible.
SEED = 21

show_setup_json(
    {
        "DATA_FRACTION": DATA_FRACTION,
        "ORIENTATION_FILE_SELECTION_MODE": ORIENTATION_FILE_SELECTION_MODE,
        "ORIENTATION_FILE_LIMIT": ORIENTATION_FILE_LIMIT,
        "TRAIN_FRACTION": TRAIN_FRACTION,
        "VALIDATION_FRACTION": VALIDATION_FRACTION,
        "SEED": SEED,
    }
)


In [ ]:
import sys

if str(NOTEBOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_ROOT))

from scripts.session1_intro_notebook_setup import build_intro_notebook_namespace

globals().update(
    build_intro_notebook_namespace(
        notebook_root=NOTEBOOK_ROOT,
        read_raw_data_dir=READ_RAW_DATA_DIR,
        read_cache_dir=READ_CACHE_DIR,
        cache_bundle_name=CACHE_BUNDLE_NAME,
        seed=SEED,
    )
)

print(
    {
        **INTRO_NOTEBOOK_SETUP_SUMMARY,
        "TASK_MODE": TASK_MODE,
        "TARGET_FLAG": TARGET_FLAG,
        "DATA_FRACTION": DATA_FRACTION,
        "SHARED_DAQATHON_ROOT": str(SHARED_DAQATHON_ROOT) if SHARED_DAQATHON_ROOT else None,
    }
)


## 🚀 FIR Input Staging

On FIR, the shared project space is the long-lived home for the data. But interactive notebook jobs on Alliance systems usually also get a directory called `$SLURM_TMPDIR`.

`SLURM_TMPDIR` is a **job-specific temporary working directory** created for you when Slurm starts a scheduled job or interactive session on a compute node. In other words:

- it appears when your Jupyter job is actually running on allocated resources,
- it lives on storage attached to that compute node,
- and it disappears when the job ends.

That makes it very different from your shared `HOME`, `SCRATCH`, or project directories:

- shared filesystems are persistent and great for canonical storage,
- `SLURM_TMPDIR` is temporary but usually much faster for repeated reads and writes during one run.

That speed matters here because notebooks repeatedly scan parquet parts, load row samples, and build model inputs. Copying the raw CSV directory and the prepared parquet cache into `SLURM_TMPDIR` once at the start usually makes the rest of the session feel much more responsive and reduces repeated traffic to the shared filesystem.

That means:

- the shared project space stays as the canonical source,
- the notebook reads from a fast local working copy during the run,
- and all generated files stay in your job-local runtime area instead of polluting the shared project tree.

One important caution: `SLURM_TMPDIR` is **not** long-term storage. Anything you want to keep after the job ends should be copied to `SCRATCH`, project storage, or a Git repo. This notebook uses `SLURM_TMPDIR` for speed, not for permanence.

The next code cell shows a progress bar while those copies are happening.


In [ ]:
# On FIR, stage shared inputs into node-local storage before we start reading them.
read_raw_data_dir = Path(READ_RAW_DATA_DIR)
runtime_raw_data_dir = Path(RUNTIME_RAW_DATA_DIR)
read_cache_dir = Path(READ_CACHE_DIR)
runtime_cache_dir = Path(RUNTIME_CACHE_DIR)

raw_stage_result = {"staged": False, "reason": "raw-data staging is disabled because SLURM_TMPDIR is not available"}
cache_stage_result = {"staged": False, "reason": "cache staging is disabled because SLURM_TMPDIR is not available"}
if USE_RUNTIME_RAW_DATA_FOR_READS:
    raw_stage_result = stage_directory_into_runtime(
        source_dir=read_raw_data_dir,
        runtime_dir=runtime_raw_data_dir,
        force=False,
        show_progress=True,
        progress_label=f"Staging raw scalar CSV files for {DATASET_PROFILE['label']} to SLURM_TMPDIR",
    )
if USE_RUNTIME_CACHE_FOR_READS:
    cache_stage_result = stage_cache_into_runtime(
        persistent_cache_dir=read_cache_dir,
        runtime_cache_dir=runtime_cache_dir,
        force=False,
        show_progress=True,
        progress_label="Staging parquet cache to SLURM_TMPDIR",
    )

active_raw_data_dir = runtime_raw_data_dir if runtime_raw_data_dir.exists() and any(runtime_raw_data_dir.glob("*.csv")) else read_raw_data_dir
active_cache_paths = choose_cache_bundle_paths([runtime_cache_dir, read_cache_dir], cache_stem=CACHE_BUNDLE_NAME)
RAW_DATA_DIR = str(active_raw_data_dir)
CACHE_DIR = str(active_cache_paths.root)
ROW_CACHE_DIR = str(active_cache_paths.row_level_dir)
WINDOW_CACHE_PATH = str(active_cache_paths.window_cache_path)
METADATA_PATH = str(active_cache_paths.metadata_path)

print(
    {
        "DATASET_PROFILE_ID": DATASET_PROFILE_ID,
        "raw_stage_result": raw_stage_result,
        "cache_stage_result": cache_stage_result,
        "active_raw_data_dir": RAW_DATA_DIR,
        "active_cache_dir": CACHE_DIR,
        "cache_bundle_name": CACHE_BUNDLE_NAME,
        "resolved_cache_stem": active_cache_paths.stem,
    }
)


## Part 1 — Orientation and Dataset

Start by understanding what the selected scalar dataset contains, what the target labels mean, and what success looks like before we touch any modelling code.


#### About The Dataset

This notebook now supports a small set of **scalar dataset presets** rather than one hard-coded instrument export.

The active preset is chosen in the configuration cell near the top. That preset tells the notebook:

- which raw-data folder to read,
- which target-label column is used,
- which measurement columns to model,
- which two variables to emphasize in plots,
- and whether k-means should cluster window summaries or row-level values.

Across all of these presets, the mental model stays the same:

- each row is one timestamp in UTC,
- measurement columns hold scalar sensor values,
- target-label columns hold the reviewed label we want the models to learn.

The printed config summary tells you exactly which preset is active for the current run.


#### What The QC Flags Mean

Many ONC scalar datasets use the standard QAQC flag system. Some presets, such as conductivity plugs, use a custom label column instead; the active target column and label meanings are shown in the setup output above. For standard QAQC flags, the official reference is:

- [ONC Quality Assurance Quality Control](https://wiki.oceannetworks.ca/spaces/DP/pages/42174414/Quality+Assurance+Quality+Control)

A practical reading of the most common flags is:

- `0`: no quality control has been applied to that value yet
- `1`: passed all tests and is treated as good data
- `2`: probably good, but worth a little more caution than `1`
- `3`: probably bad or suspect
- `4`: bad and usually treated as failed data
- `6`: insufficient valid data for reliable down-sampling
- `7`: averaged value
- `8`: interpolated value
- `9`: missing value (`NaN`)

A few practical notes are especially important for this notebook:

- `1` is the clearest "normal / healthy" label.
- `3` and `4` are the main issue labels we usually care about in supervised QC experiments.
- `9` means the value is missing, not merely noisy.
- `0` does **not** mean good; it means no QC decision has been applied.

ONC's reference page also explains that manual review can override automated tests, which is useful context when you see the same instrument behaviour receive different final flags over time.

The selected target flag column is shown in the config summary, and later cells will show which QC values actually appear in the active dataset and which ones we treat as reviewed labels for modelling.


### Looking At Real Examples Early

Before moving on to caching or feature engineering, it helps to inspect a lightweight sample of the raw CSV data directly.

In the next few cells we:

- load a small sample from a few raw CSV files spread across the time range,
- look at a few real rows,
- and plot representative QC intervals in context.

This gives you a concrete picture of what one timestamp looks like before the notebook starts building model inputs from it.


### Small Utilities For Exploration Samples

A few small file-reading helpers live in `scripts/session1_intro_utils.py` so this notebook can stay focused on the data itself.

That keeps the early orientation cells simple while leaving the split and train-subsampling logic visible later in Part 3.


In [ ]:
# Load a lightweight exploration sample directly from raw CSV files.
raw_csv_paths = sorted(Path(RAW_DATA_DIR).glob("*.csv"))
if not raw_csv_paths:
    raise FileNotFoundError(f"No CSV files found in {RAW_DATA_DIR}")

# Conductivity-plug examples should always be selected and coloured by the
# reviewed ML label, not by the raw instrument QAQC columns.
FLAG_EXAMPLE_TARGET = "ml_label" if DATASET_PROFILE_ID == "conductivity_plugs" else TARGET_FLAG
FLAG_EXAMPLE_DISPLAY_NAME = "ml_label" if DATASET_PROFILE_ID == "conductivity_plugs" else "QC flag"
FLAG_EXAMPLE_LABEL_MEANINGS = (
    {
        0: "good",
        1: "conductivity bad plug",
        2: "conductivity bad other",
        3: "non-conductivity failure",
        4: "missing data",
    }
    if DATASET_PROFILE_ID == "conductivity_plugs"
    else None
)
FLAG_EXAMPLE_AVOID_CONTEXT_LABELS = (4,) if DATASET_PROFILE_ID == "conductivity_plugs" else (9,)

orientation_required_columns = ["Time UTC", FLAG_EXAMPLE_TARGET]
if PLOT_MEASUREMENT_COLUMN:
    orientation_required_columns.append(PLOT_MEASUREMENT_COLUMN)
orientation_candidate_paths = filter_csv_paths_with_required_columns(
    raw_csv_paths,
    orientation_required_columns,
)
orientation_source_paths = orientation_candidate_paths if orientation_candidate_paths else raw_csv_paths

orientation_selected_paths = select_part_paths(
    orientation_source_paths,
    limit=ORIENTATION_FILE_LIMIT,
    mode=ORIENTATION_FILE_SELECTION_MODE,
)
ORIENTATION_ROWS_PER_FILE = 4000
orientation_columns = list(
    dict.fromkeys(
        [
            "Time UTC",
            FLAG_EXAMPLE_TARGET,
            TARGET_FLAG,
            *OPTIONAL_QC_COLUMNS,
            PLOT_MEASUREMENT_COLUMN,
            PLOT_SECONDARY_COLUMN,
            *MEASUREMENT_COLUMNS,
            "Depth (m)",
        ]
    )
)

exploration_df = load_raw_scalar_sample(
    orientation_selected_paths,
    sample_rows_per_file=ORIENTATION_ROWS_PER_FILE,
    columns=orientation_columns,
)

flag_context_df = load_raw_flag_context_sample(
    orientation_selected_paths,
    target_flag=FLAG_EXAMPLE_TARGET,
    classes=FLAG_EXAMPLE_CLASSES,
    context_rows_per_class=BASE_FLAG_EXAMPLE_POINTS_PER_PANEL,
    columns=orientation_columns,
)

print(
    {
        "orientation_raw_data_dir": RAW_DATA_DIR,
        "orientation_candidate_files": len(orientation_candidate_paths),
        "orientation_selected_files": len(orientation_selected_paths),
        "orientation_rows_loaded": len(exploration_df),
        "flag_context_rows_loaded": len(flag_context_df),
        "FLAG_EXAMPLE_TARGET": FLAG_EXAMPLE_TARGET,
        "FLAG_EXAMPLE_DISPLAY_NAME": FLAG_EXAMPLE_DISPLAY_NAME,
        "FLAG_EXAMPLE_AVOID_CONTEXT_LABELS": FLAG_EXAMPLE_AVOID_CONTEXT_LABELS,
        "ORIENTATION_FILE_SELECTION_MODE": ORIENTATION_FILE_SELECTION_MODE,
        "ORIENTATION_FILE_LIMIT": ORIENTATION_FILE_LIMIT,
        "ORIENTATION_ROWS_PER_FILE": ORIENTATION_ROWS_PER_FILE,
        "orientation_preview_columns": orientation_columns,
    }
)




In [ ]:
# Look at a few rows so the dataset description is tied to the actual table.
preview_candidates = [
    "Time UTC",
    PLOT_MEASUREMENT_COLUMN,
    PLOT_SECONDARY_COLUMN,
    "Depth (m)",
    TARGET_FLAG,
    *OPTIONAL_QC_COLUMNS,
]
preview_columns = [column for column in preview_candidates if column in exploration_df.columns]
display(exploration_df[preview_columns].head(8))
print(
    {
        "time_start": exploration_df["Time UTC"].min().isoformat(),
        "time_end": exploration_df["Time UTC"].max().isoformat(),
        "preview_columns": preview_columns,
    }
)


### Seeing The Target Labels In Context

Before training anything, it helps to look at the time series directly.

The plots below show local windows centred on representative target-label values pulled directly from the raw CSV files. This is useful because:

- it connects the abstract label numbers to real sensor behaviour,
- it shows that some problematic labels appear in coherent stretches rather than isolated single rows,
- it reminds you that the model is trying to learn from patterns in time, not from labels in isolation.

If you want to make these panels wider or more annotated, the next config cell exposes:

- `FLAG_EXAMPLE_POINTS_PER_PANEL`
- `FLAG_EXAMPLE_REGION_ALPHA`
- `FLAG_EXAMPLE_SHOW_POINTS`
- `FLAG_EXAMPLE_CLASSES`


In [ ]:
FLAG_EXAMPLE_POINTS_PER_PANEL = 5_000 if DATASET_PROFILE_ID == "conductivity_plugs" else BASE_FLAG_EXAMPLE_POINTS_PER_PANEL
FLAG_EXAMPLE_REGION_ALPHA = 0.18
FLAG_EXAMPLE_SHOW_POINTS = DATASET_PROFILE_ID == "conductivity_plugs"

print(
    {
        "FLAG_EXAMPLE_POINTS_PER_PANEL": FLAG_EXAMPLE_POINTS_PER_PANEL,
        "FLAG_EXAMPLE_REGION_ALPHA": FLAG_EXAMPLE_REGION_ALPHA,
        "FLAG_EXAMPLE_SHOW_POINTS": FLAG_EXAMPLE_SHOW_POINTS,
        "FLAG_EXAMPLE_CLASSES": FLAG_EXAMPLE_CLASSES,
    }
)




In [ ]:
# Plot representative local windows for the main target-label values we see in this dataset.
timeseries_figure, timeseries_examples = plot_flag_examples(
    flag_context_df,
    target_flag=FLAG_EXAMPLE_TARGET,
    target_display_name=FLAG_EXAMPLE_DISPLAY_NAME,
    label_meanings=FLAG_EXAMPLE_LABEL_MEANINGS,
    avoid_context_labels=FLAG_EXAMPLE_AVOID_CONTEXT_LABELS,
    measurement_column=PLOT_MEASUREMENT_COLUMN,
    secondary_column=PLOT_SECONDARY_COLUMN,
    points_per_panel=FLAG_EXAMPLE_POINTS_PER_PANEL,
    classes=FLAG_EXAMPLE_CLASSES,
    good_labels=GOOD_LABELS,
    region_alpha=FLAG_EXAMPLE_REGION_ALPHA,
    show_flag_points=FLAG_EXAMPLE_SHOW_POINTS,
)
plt.show()
display(timeseries_examples)




## Part 2 — Data Preparation and Caching

This part explains how to turn raw ONC CSV exports into typed parquet tables that are much easier to reuse in ML experiments.


### 🧹 How We Prepare The Data

This section keeps the preparation story short and reusable:

1. start from raw ONC CSV exports,
2. clean them into typed rows,
3. save a reusable parquet cache,
4. reuse that cache for the rest of the notebook.

The detailed parsing logic lives in `scripts/prepare_scalar_session1_data.py`. Here we focus on the inputs, outputs, and the few knobs you are most likely to change.


### Step-By-Step: Turning Raw Data Into ML-Ready Data

A good preparation pipeline is usually simpler than it first sounds.

**1. Decide what one example means**

For this notebook, one cleaned row is the reusable base unit. Later sections regroup those rows into windows when a model needs temporal context.

**2. Keep an explicit list of the fields you need**

For scalar time series, that usually means:

- a reliable timestamp,
- one or more measurement columns,
- the target-label column,
- any extra context columns,
- and an identifier such as `source_file`.

**3. Clean and type the raw values**

In the prep script we:

- locate the real ONC header row,
- parse `Time UTC` into datetimes,
- convert measurement and QC columns into numeric values,
- drop unusable timestamps,
- sort by time,
- and remember which file each row came from.

**4. Save reusable artifacts**

We keep two views of the same cleaned data:

- **row-level parquet**: one row per timestamp, useful for tabular models and sequence building,
- **window-summary parquet**: one row per fixed window, useful when you want short stretches summarised into one feature vector.


---


### 📦 Build Or Reuse The Parquet Cache

Most of the notebook uses a prepared parquet cache. Rebuilding it is optional when a shared or previously generated cache is already available.

When we do rebuild it, there are two separate outputs:

- `create_session1_row_level_parquet_cache(...)`: reads raw CSV files, cleans and types the rows, aligns companion device streams when needed, and writes row-level parquet parts.
- `create_session1_window_level_parquet_cache(...)`: reads those row-level parquet parts and writes one window-summary parquet file for k-means and short-window exploration.

For ordinary CSVs that only need a simple row-level cache, `session1_intro_utils.csv_files_to_parquet_cache(...)` is the smaller helper. It supports `header="auto"`, `header="first_row"`, or an integer header row such as `header=5`.


In [ ]:
RUN_RAW_PREP = False
FORCE_REBUILD_CACHE = False
PREP_MAX_FILES = None
PREP_SAMPLE_ROWS = None
PREP_WINDOW_SIZE = 256
PREP_MERGE_TOLERANCE_SECONDS = 5

show_setup_json(
    {
        "RUN_RAW_PREP": RUN_RAW_PREP,
        "FORCE_REBUILD_CACHE": FORCE_REBUILD_CACHE,
        "PREP_MAX_FILES": PREP_MAX_FILES,
        "PREP_SAMPLE_ROWS": PREP_SAMPLE_ROWS,
        "PREP_WINDOW_SIZE": PREP_WINDOW_SIZE,
        "PREP_MERGE_TOLERANCE_SECONDS": PREP_MERGE_TOLERANCE_SECONDS,
    }
)


In [ ]:
# The row-level cache keeps one cleaned row per timestamp.
# The window-level cache summarises those rows into fixed-size windows for k-means.
# RUNTIME_CACHE_DIR is where this notebook writes new cache files during this run.
runtime_cache_path = Path(RUNTIME_CACHE_DIR)
# Prefer a cache from this run, but fall back to READ_CACHE_DIR if a prepared cache already exists there.
existing_cache_paths = choose_cache_bundle_paths(
    [runtime_cache_path, Path(READ_CACHE_DIR)],
    # CACHE_BUNDLE_NAME names the row/window/metadata files for the active dataset.
    cache_stem=CACHE_BUNDLE_NAME,
)
cache_metadata_path = existing_cache_paths.metadata_path
# RUN_RAW_PREP rebuilds on demand; FORCE_REBUILD_CACHE is for intentionally overwriting stale cache files.
should_run_prep = FORCE_REBUILD_CACHE or RUN_RAW_PREP

show_setup_json(
    {
        "should_run_prep": should_run_prep,
        "cache_exists": cache_metadata_path.exists(),
        "raw_data_dir": RAW_DATA_DIR,
        "cache_bundle_name": CACHE_BUNDLE_NAME,
        "row_level_step": "raw CSV files -> typed row-level parquet parts",
        "window_level_step": "row-level parquet parts -> fixed-window summary parquet",
    }
)

if should_run_prep:
    row_cache_result = create_session1_row_level_parquet_cache(
        # raw_data_dir: folder containing the source CSV files.
        raw_data_dir=RAW_DATA_DIR,
        # cache_root: output folder where the parquet bundle is written.
        cache_root=runtime_cache_path,
        # cache_bundle_name: shared filename stem for row parts, window file, and metadata JSON.
        cache_bundle_name=CACHE_BUNDLE_NAME,
        # target_flag: reviewed QC/custom label column used as the ML target later.
        target_flag=TARGET_FLAG,
        # primary_device: dataset-specific stream to treat as the main timestamp grid.
        primary_device=PRIMARY_DEVICE,
        # measurement_columns: sensor-value columns kept for plots and ML features.
        measurement_columns=MEASUREMENT_COLUMNS,
        # max_files: optional quick-run limit; None means read every CSV in RAW_DATA_DIR.
        max_files=PREP_MAX_FILES,
        # sample_rows: optional per-file row limit; None means keep all rows from each CSV.
        sample_rows=PREP_SAMPLE_ROWS,
        # merge_tolerance_seconds: maximum timestamp mismatch allowed when aligning companion streams.
        merge_tolerance_seconds=PREP_MERGE_TOLERANCE_SECONDS,
    )
    show_setup_json(row_cache_result["summary"])

    window_cache_result = create_session1_window_level_parquet_cache(
        # row_cache_result tells the window builder where the cleaned row-level parts were written.
        row_cache_result,
        # target_flag and issue_labels let each window store its issue rate.
        target_flag=TARGET_FLAG,
        issue_labels=ISSUE_LABELS,
        # window_size is the number of row-level timestamps summarised into one window.
        window_size=PREP_WINDOW_SIZE,
        # The remaining arguments keep the window metadata consistent with the row-level cache.
        sample_rows=PREP_SAMPLE_ROWS,
        merge_tolerance_seconds=PREP_MERGE_TOLERANCE_SECONDS,
        primary_device=PRIMARY_DEVICE,
        max_files=PREP_MAX_FILES,
    )
    show_setup_json(window_cache_result["summary"])
elif cache_metadata_path.exists():
    print("Using the existing prepared cache. Set RUN_RAW_PREP = True if you want to rebuild it.")
else:
    print(
        "No prepared cache was found. Point READ_CACHE_DIR at a prebuilt cache, or set RUN_RAW_PREP = True "
        "to build row-level and window-level parquet files from the raw CSV files."
    )


### Where The Prep Code Lives

Use `session1_intro_utils.csv_files_to_parquet_cache(...)` when you just need a row-level parquet cache from one or more CSV files.

This workshop uses `scripts/prepare_scalar_session1_data.py` for the full cache because it also handles dataset-specific alignment and the window-summary parquet used later for clustering.

A useful reading order is:

1. `main()` for the overall flow,
2. `read_scalar_csv(...)` for row cleaning and typing,
3. `build_window_features(...)` for the window-summary cache.


---


### ⚡ Why The Parquet Cache Helps

Raw CSV is the source format. Parquet is the reusable analysis format.

This section shows four things:

- which cache bundle is active,
- which columns this notebook will read from the row-level cache,
- how the row-level and window-summary caches differ,
- how read time changes when we read all columns versus only the selected notebook columns.

For small, compact CSV samples, a full parquet read may not beat CSV. The practical benefit is that parquet keeps typed columns and supports **column-pruned reads**: loading only the timestamp, source file, target label, optional QC/context fields, and measurement columns needed by the next analysis step.


In [ ]:
cache_context = show_session1_cache_inspection(
    raw_data_dir=RAW_DATA_DIR,
    runtime_cache_dir=RUNTIME_CACHE_DIR,
    read_cache_dir=READ_CACHE_DIR,
    cache_bundle_name=CACHE_BUNDLE_NAME,
    target_flag=TARGET_FLAG,
    measurement_columns=MEASUREMENT_COLUMNS,
    optional_qc_columns=OPTIONAL_QC_COLUMNS,
    plot_measurement_column=PLOT_MEASUREMENT_COLUMN,
    plot_secondary_column=PLOT_SECONDARY_COLUMN,
)

# Later cells use these resolved cache paths and active column lists.
globals().update(cache_context["notebook_values"])


In [ ]:
PARQUET_READ_SAMPLE_ROWS = 50_000

cache_read_result = show_session1_cache_read_comparison(
    representative_raw_path=representative_raw_path,
    representative_parquet_path=representative_parquet_path,
    row_use_columns=ROW_USE_COLUMNS,
    sample_rows=PARQUET_READ_SAMPLE_ROWS,
)


### Choosing A Useful Storage Format

Parquet is a strong fit for **typed tabular data** and many **1D time-series** workflows, but it is not the only good option.

A simple rule of thumb is: store the data in a format that matches the natural unit of your experiment.

- **Scalar tables / 1D time series**: parquet is often ideal because you can read only the columns you need.
- **Sequences that will be windowed later**: row-level parquet still works well because you can build windows on demand.
- **Images or 2D products** such as spectrograms: a folder of image files can be perfectly reasonable if each file is already one example.
- **Large 2D or 3D numeric arrays**: formats like NumPy `.npy` / `.npz`, HDF5, Zarr, or NetCDF are often a better fit than CSV.
- **Text examples**: JSONL and parquet are common because each row can hold one document plus labels and metadata.

So the real lesson is not "always use parquet." It is "pick a reusable format that preserves the structure of your examples and is efficient for the experiments you plan to run."


## Part 3 — Understanding and Defining the Reviewed Modelling Dataset

This part turns the prepared parquet cache into the data the models will use.

In this notebook, the **reviewed modelling dataset** means: the rows with usable reviewed labels, after any row-budget cap from `DATA_FRACTION`, plus the model target columns we create from those labels.

The **fixed split** means: the train / validation / test division that stays unchanged while we compare different models and train-only sampling choices.

The code uses the same vocabulary: `reviewed_model_df` is the labelled row dataset used for modelling, and `FIXED_SPLIT_STRATEGY` describes how those rows are divided into train, validation, and test.


### What This Part Is For

Now that the data is cleaned and stored efficiently, the next job is to define the reviewed modelling dataset and fixed evaluation split.

In this part you will:

- choose which prepared parquet files to inspect,
- keep only the reviewed rows,
- compare train / validation / test split strategies,
- lock one fixed split,
- then optionally shrink only the training split.

A lot of the clarity in an ML workflow comes from this dataset-definition step, not just from the model choice later on.


### Phase 1 — Define The Reviewed Modelling Dataset

In this first phase, we use the prepared parquet cache to understand the reviewed data, compare fixed split strategies, and lock the train / validation / test split before we make any train-only runtime decisions.


---


In [ ]:
# Load the metadata that was written during cache preparation.
persistent_cache_dir = Path(READ_CACHE_DIR)
runtime_cache_dir = Path(RUNTIME_CACHE_DIR)
active_cache_paths = choose_cache_bundle_paths([runtime_cache_dir, persistent_cache_dir], cache_stem=CACHE_BUNDLE_NAME)
CACHE_DIR = str(active_cache_paths.root)
ROW_CACHE_DIR = str(active_cache_paths.row_level_dir)
WINDOW_CACHE_PATH = str(active_cache_paths.window_cache_path)
METADATA_PATH = str(active_cache_paths.metadata_path)

row_cache_dir = active_cache_paths.row_level_dir
window_cache_path = active_cache_paths.window_cache_path
metadata_path = active_cache_paths.metadata_path

if not metadata_path.exists():
    raise FileNotFoundError(
        f"Missing cache metadata at {metadata_path}. Run scripts/prepare_scalar_session1_data.py first."
    )

metadata = json.loads(metadata_path.read_text())
part_paths = sorted(row_cache_dir.glob("*.parquet"))
if not part_paths:
    raise FileNotFoundError(f"No parquet parts found in {row_cache_dir}")

part_to_source = {
    Path(file_info["row_level_part"]).name: file_info["source_file"]
    for file_info in metadata["processed_files"]
}
source_to_row_part = {
    file_info["source_file"]: row_cache_dir / Path(file_info["row_level_part"]).name
    for file_info in metadata["processed_files"]
}

print(
    {
        "active_cache_dir": str(active_cache_paths.root),
        "cache_bundle_name": CACHE_BUNDLE_NAME,
        "runtime_output_root": str(RUNTIME_OUTPUT_ROOT),
        "model_output_dir": str(MODEL_OUTPUT_DIR),
        "row_cache_parts": len(part_paths),
        "window_cache_path": str(window_cache_path),
    }
)


In [ ]:
# Summarise the overall prepared cache before we define the reviewed modelling dataset.
cache_summary = {
    "full_row_count": metadata["row_count"],
    "full_window_count": metadata["window_count"],
    "processed_file_count": metadata["processed_file_count"],
    "issue_fraction": round(float(metadata["issue_fraction"]), 4),
    "task_mode": TASK_MODE,
}
print(cache_summary)

processed_file_summary = pd.DataFrame(metadata["processed_files"])
processed_file_summary = processed_file_summary[["source_file", "row_count", "time_start", "time_end"]].copy()
processed_file_summary["source_file"] = processed_file_summary["source_file"].str.replace(
    r"_\d{8}T\d{6}Z_\d{8}T\d{6}Z-NaN\.csv$",
    "",
    regex=True,
)
display(processed_file_summary.head(8))


#### 🚚 Use The Prepared Cache To Understand The Dataset

Parquet gives us fast access to the cleaned dataset, so we can inspect prepared data directly with only the columns we need.

In this part we will:

- load the prepared parquet cache,
- summarise reviewed vs unreviewed rows,
- check how the flags change across time,
- compare split strategies on the reviewed rows,
- then choose one fixed split.


#### 🎚️ Make The Row Budgets Visible

Large reviewed datasets can make live notebook runs slow. This section shows the exact row/window caps used to keep the workshop runnable.

`DATA_FRACTION` is the single top-level size control. Set `USE_DATA_FRACTION_BUDGETS = False` in the next cell to keep all reviewed rows and remove the per-section row/window caps.

| Cap variable | Where it is used | How it is computed when caps are on | Why it exists |
|---|---|---|---|
| `REVIEWED_MODEL_ROW_LIMIT` | reviewed modelling dataframe | `FULL_REVIEWED_ROW_COUNT * DATA_FRACTION` | build a smaller reviewed dataframe before split comparison |
| `TRAIN_SUBSET_MAX_ROWS` | train-only subset comparison | `TRAIN_SUBSET_BASE_ROWS * DATA_FRACTION`, with a minimum | compare train-subsampling strategies quickly |
| `ISSUE_ROWS_PER_FILE` | `issue_focused` train subset | `ISSUE_ROWS_PER_FILE_BASE * DATA_FRACTION`, with a minimum | control how many extra issue rows are added |
| `RF_TRAIN_MAX_ROWS` | Random Forest fit | `RF_TRAIN_BASE_ROWS * DATA_FRACTION`, with a minimum | keep the Random Forest baseline fast enough for a live run |
| `RF_VALIDATION_MAX_ROWS` / `RF_TEST_MAX_ROWS` | Random Forest scoring | `RF_EVAL_BASE_ROWS * DATA_FRACTION`, with a minimum | score representative validation/test rows without loading every row |
| `KMEANS_WINDOW_LIMIT` | k-means rows/windows | `KMEANS_WINDOW_BASE_ROWS * DATA_FRACTION`, with a minimum | keep clustering and example plots readable and fast |


In [ ]:
# Use every parquet part from the prepared cache. DATA_FRACTION limits rows later; it does not choose files.
TEMPORAL_SUMMARY_BIN_COUNT = 24
selected_paths = part_paths
# Keep the source-file names handy for summaries and cache inspection.
selected_source_files = {part_to_source[path.name] for path in selected_paths}

# Set this to False when you want full-data modelling instead of quicker workshop-sized runs.
USE_DATA_FRACTION_BUDGETS = True
if not 0 < DATA_FRACTION <= 1:
    raise ValueError("DATA_FRACTION must be in the interval (0, 1].")

# Count the rows that have one of the labels we will use for modelling.
# This lets us show the reviewed-row cap before we build the modelling dataframe.
target_distribution = {int(label): int(count) for label, count in metadata.get("target_distribution", {}).items()}
FULL_REVIEWED_ROW_COUNT = sum(
    target_distribution.get(int(label), 0)
    for label in [*GOOD_LABELS, *ISSUE_LABELS]
)

# Base caps are the row/window limits used when DATA_FRACTION = 1.0 but caps are still on.
# Minimums prevent very small DATA_FRACTION values from producing unusably tiny examples.
TRAIN_SUBSET_BASE_ROWS = 1_000_000
TRAIN_SUBSET_MIN_ROWS = 10_000
ISSUE_ROWS_PER_FILE_BASE = 12_000
ISSUE_ROWS_PER_FILE_MIN = 1_000
RF_TRAIN_BASE_ROWS = 250_000
RF_TRAIN_MIN_ROWS = 25_000
RF_EVAL_BASE_ROWS = 150_000
RF_EVAL_MIN_ROWS = 15_000
KMEANS_WINDOW_BASE_ROWS = 5_000
KMEANS_WINDOW_MIN_ROWS = 2_000

if USE_DATA_FRACTION_BUDGETS and DATA_FRACTION < 0.999:
    BUDGET_MODE = "DATA_FRACTION row/window caps"
    BUDGET_DATA_FRACTION = DATA_FRACTION
    # This is the main dataset-size control: keep this fraction of usable reviewed rows.
    REVIEWED_MODEL_ROW_LIMIT = max(1, int(round(FULL_REVIEWED_ROW_COUNT * DATA_FRACTION)))
    # The remaining caps keep specific model/demo sections fast and memory-safe.
    TRAIN_SUBSET_MAX_ROWS = max(TRAIN_SUBSET_MIN_ROWS, int(TRAIN_SUBSET_BASE_ROWS * DATA_FRACTION))
    ISSUE_ROWS_PER_FILE = max(ISSUE_ROWS_PER_FILE_MIN, int(ISSUE_ROWS_PER_FILE_BASE * DATA_FRACTION))
    RF_TRAIN_MAX_ROWS = max(RF_TRAIN_MIN_ROWS, int(RF_TRAIN_BASE_ROWS * DATA_FRACTION))
    RF_VALIDATION_MAX_ROWS = max(RF_EVAL_MIN_ROWS, int(RF_EVAL_BASE_ROWS * DATA_FRACTION))
    RF_TEST_MAX_ROWS = max(RF_EVAL_MIN_ROWS, int(RF_EVAL_BASE_ROWS * DATA_FRACTION))
    KMEANS_WINDOW_LIMIT = max(KMEANS_WINDOW_MIN_ROWS, int(KMEANS_WINDOW_BASE_ROWS * DATA_FRACTION))
else:
    BUDGET_MODE = "full reviewed data / no row caps"
    BUDGET_DATA_FRACTION = 1.0
    REVIEWED_MODEL_ROW_LIMIT = FULL_REVIEWED_ROW_COUNT
    TRAIN_SUBSET_MAX_ROWS = None
    ISSUE_ROWS_PER_FILE = ISSUE_ROWS_PER_FILE_BASE
    RF_TRAIN_MAX_ROWS = None
    RF_VALIDATION_MAX_ROWS = None
    RF_TEST_MAX_ROWS = None
    KMEANS_WINDOW_LIMIT = None

# Summarize the source files represented by the selected parquet parts.
selected_file_summary = pd.DataFrame(metadata["processed_files"])
selected_file_summary = selected_file_summary[selected_file_summary["source_file"].isin(selected_source_files)].reset_index(drop=True)
selected_file_summary = selected_file_summary[["source_file", "row_count", "time_start", "time_end"]].copy()
selected_file_summary["source_file"] = selected_file_summary["source_file"].str.replace(
    r"_\d{8}T\d{6}Z_\d{8}T\d{6}Z-NaN\.csv$",
    "",
    regex=True,
)
display(selected_file_summary.head(8))

print(
    {
        "DATA_FRACTION": DATA_FRACTION,
        "USE_DATA_FRACTION_BUDGETS": USE_DATA_FRACTION_BUDGETS,
        "BUDGET_MODE": BUDGET_MODE,
        "FULL_REVIEWED_ROW_COUNT": FULL_REVIEWED_ROW_COUNT,
        "REVIEWED_MODEL_ROW_LIMIT": REVIEWED_MODEL_ROW_LIMIT,
        "TRAIN_SUBSET_MAX_ROWS": TRAIN_SUBSET_MAX_ROWS,
        "RF_TRAIN_MAX_ROWS": RF_TRAIN_MAX_ROWS,
        "RF_VALIDATION_MAX_ROWS": RF_VALIDATION_MAX_ROWS,
        "RF_TEST_MAX_ROWS": RF_TEST_MAX_ROWS,
        "KMEANS_WINDOW_LIMIT": KMEANS_WINDOW_LIMIT,
        "selected_parts": len(selected_paths),
        "selected_source_files": len(selected_source_files),
        "TEMPORAL_SUMMARY_BIN_COUNT": TEMPORAL_SUMMARY_BIN_COUNT,
    }
)


#### 🧰 Split And Subsampling Helpers

The next cell defines the helper functions used to build the fixed split and the train-only subsets.

Read these as examples of simple time-aware split and subsampling patterns that you can adapt for your own data.


In [ ]:
# These helpers define the split and train-only sampling logic used in Part 3.
# Each function focuses on one idea so you can trace how the rows move through the workflow.

def evenly_spaced_take(frame: pd.DataFrame, limit: int | None, *, time_column: str = "Time UTC") -> pd.DataFrame:
    """Return up to `limit` rows spread across the dataframe's full time range.

    Use `time_column="Time UTC"` for row-level data and
    `time_column="window_start"` for cached window-summary data.
    """
    # Sort by the relevant time column so the sample keeps chronological order.
    sort_column = time_column if time_column in frame.columns else "window_start" if "window_start" in frame.columns else None
    ordered = frame.sort_values(sort_column).reset_index(drop=True).copy() if sort_column else frame.reset_index(drop=True).copy()
    # If we do not need to shrink the dataframe, return the full ordered version.
    if limit is None or len(ordered) <= limit:
        return ordered
    # Pick row positions spread across the whole time range instead of keeping only the earliest rows.
    indices = np.linspace(0, len(ordered) - 1, num=limit, dtype=int)
    return ordered.iloc[indices].reset_index(drop=True)


def split_global_contiguous(
    frame: pd.DataFrame,
    *,
    train_fraction: float,
    validation_fraction: float,
) -> dict[str, pd.DataFrame]:
    """Split one dataframe into early train, middle validation, and late test segments.

    Use this when you want each split to cover a different part of the full
    fixed-split timeline.
    """
    # This strategy makes one early / middle / late cut across the full time axis.
    ordered = frame.sort_values("Time UTC").reset_index(drop=True).copy()
    train_cut = int(len(ordered) * train_fraction)
    validation_cut = int(len(ordered) * (train_fraction + validation_fraction))
    return {
        "train": ordered.iloc[:train_cut].reset_index(drop=True),
        "validation": ordered.iloc[train_cut:validation_cut].reset_index(drop=True),
        "test": ordered.iloc[validation_cut:].reset_index(drop=True),
    }


def split_per_source_contiguous(
    frame: pd.DataFrame,
    *,
    train_fraction: float,
    validation_fraction: float,
    source_column: str = "source_file",
) -> dict[str, pd.DataFrame]:
    """Apply the contiguous time split within each source file, then recombine the results.

    This lets every source contribute rows to train, validation, and test while
    still preserving time order inside each file.
    """
    # If we do not know which file each row came from, fall back to one global time split.
    if source_column not in frame.columns:
        return split_global_contiguous(
            frame,
            train_fraction=train_fraction,
            validation_fraction=validation_fraction,
        )

    # Split each source file separately so every file can contribute rows to every split.
    split_frames: dict[str, list[pd.DataFrame]] = {"train": [], "validation": [], "test": []}
    for _, source_frame in frame.groupby(source_column, sort=False, observed=False):
        local_split = split_global_contiguous(
            source_frame,
            train_fraction=train_fraction,
            validation_fraction=validation_fraction,
        )
        for split_name, split_frame in local_split.items():
            split_frames[split_name].append(split_frame)

    # Concatenate the per-file slices back together and restore one global time ordering.
    empty_template = frame.iloc[0:0].copy()
    return {
        split_name: (
            pd.concat(split_frames[split_name], ignore_index=True).sort_values("Time UTC").reset_index(drop=True)
            if split_frames[split_name]
            else empty_template.copy()
        )
        for split_name in ["train", "validation", "test"]
    }


def split_interleaved_blocks(
    frame: pd.DataFrame,
    *,
    train_fraction: float,
    validation_fraction: float,
    block_rows: int,
    source_column: str = "source_file",
) -> dict[str, pd.DataFrame]:
    """Rotate fixed-size time blocks across train, validation, and test splits.

    This mixes time periods more than one large contiguous cut while still
    keeping nearby rows together inside each block.
    """
    if block_rows <= 0:
        raise ValueError("block_rows must be positive")

    # Start from time-ordered rows so each block represents one local stretch of time.
    ordered = frame.sort_values("Time UTC").reset_index(drop=True).copy()
    if source_column not in ordered.columns:
        ordered_groups = [(None, ordered)]
    else:
        ordered_groups = list(ordered.groupby(source_column, sort=False, observed=False))

    # The cycle decides how often each split appears as we rotate through blocks.
    cycle_length = 20
    train_slots = max(1, int(round(train_fraction * cycle_length)))
    validation_slots = max(1, int(round(validation_fraction * cycle_length)))
    if train_slots + validation_slots >= cycle_length:
        validation_slots = max(1, cycle_length - train_slots - 1)

    # Walk through the files in order, cut each one into fixed-size blocks, and rotate those blocks across splits.
    split_frames: dict[str, list[pd.DataFrame]] = {"train": [], "validation": [], "test": []}
    block_index = 0
    for _, source_frame in ordered_groups:
        source_frame = source_frame.sort_values("Time UTC").reset_index(drop=True)
        for start in range(0, len(source_frame), block_rows):
            block = source_frame.iloc[start : start + block_rows].copy()
            if block.empty:
                continue
            slot = block_index % cycle_length
            if slot < train_slots:
                split_name = "train"
            elif slot < train_slots + validation_slots:
                split_name = "validation"
            else:
                split_name = "test"
            split_frames[split_name].append(block)
            block_index += 1

    # Recombine the assigned blocks and restore chronological order inside each split.
    empty_template = ordered.iloc[0:0].copy()
    return {
        split_name: (
            pd.concat(split_frames[split_name], ignore_index=True).sort_values("Time UTC").reset_index(drop=True)
            if split_frames[split_name]
            else empty_template.copy()
        )
        for split_name in ["train", "validation", "test"]
    }


def split_reviewed_frame(
    frame: pd.DataFrame,
    *,
    train_fraction: float,
    validation_fraction: float,
    strategy: str,
    block_rows: int = 1024,
) -> dict[str, pd.DataFrame]:
    """Run the selected fixed split strategy and return the three split dataframes.

    Later cells call this helper so the strategy choice lives in one place.
    """
    # This wrapper keeps the later notebook cells simple: they only need to name the strategy they want.
    if strategy == "global_contiguous":
        return split_global_contiguous(
            frame,
            train_fraction=train_fraction,
            validation_fraction=validation_fraction,
        )
    if strategy == "per_source_contiguous":
        return split_per_source_contiguous(
            frame,
            train_fraction=train_fraction,
            validation_fraction=validation_fraction,
        )
    if strategy == "interleaved_blocks":
        return split_interleaved_blocks(
            frame,
            train_fraction=train_fraction,
            validation_fraction=validation_fraction,
            block_rows=block_rows,
        )
    raise ValueError(f"Unsupported split strategy: {strategy}")


def subsample_time_spread(frame: pd.DataFrame, rows_limit: int | None) -> pd.DataFrame:
    """Shrink the training set by keeping rows spread across the full training time range."""
    # Keep rows spread across the full time range while shrinking the training set.
    return evenly_spaced_take(frame, rows_limit)


def subsample_issue_focused(
    frame: pd.DataFrame,
    *,
    rows_limit: int | None,
    target_column: str,
    issue_labels: list[int] | tuple[int, ...],
    issue_rows: int,
) -> pd.DataFrame:
    """Keep broad time coverage, then add extra reviewed issue rows to the subset.

    Use this when issue examples are rare and you want a smaller training set to
    include more of them.
    """
    # Sort first so the subset keeps a readable time ordering.
    work = frame.sort_values("Time UTC").reset_index(drop=True).copy()
    if rows_limit is None or len(work) <= rows_limit:
        return work

    # Keep a broad time-spread sample, then add extra issue examples on top of it.
    normalized_issue_labels = [int(label) for label in issue_labels]
    dedupe_columns = [column for column in ["source_file", "Time UTC"] if column in work.columns]

    base_limit = max(int(rows_limit) - int(issue_rows), 0)
    sampled_frame = evenly_spaced_take(work, base_limit)
    issue_mask = pd.to_numeric(work[target_column], errors="coerce").isin(normalized_issue_labels)
    issue_frame = work.loc[issue_mask].reset_index(drop=True)
    issue_sample = evenly_spaced_take(issue_frame, int(issue_rows)) if int(issue_rows) > 0 else issue_frame.iloc[0:0].copy()

    # Drop duplicates because some rows may appear in both the base sample and the issue-focused sample.
    sampled_frame = pd.concat([sampled_frame, issue_sample], ignore_index=True)
    if dedupe_columns:
        sampled_frame = sampled_frame.drop_duplicates(subset=dedupe_columns)
    else:
        sampled_frame = sampled_frame.drop_duplicates()
    sampled_frame = sampled_frame.sort_values("Time UTC").reset_index(drop=True)
    return evenly_spaced_take(sampled_frame, rows_limit)


def subsample_balanced_reviewed(
    frame: pd.DataFrame,
    *,
    rows_limit: int | None,
    target_column: str,
    good_labels: list[int] | tuple[int, ...],
    issue_labels: list[int] | tuple[int, ...],
    balanced_issue_share: float,
) -> pd.DataFrame:
    """Build a smaller training set with a chosen good-versus-issue class mix.

    The requested issue share is a target, not a guarantee. If the full training
    split contains fewer issue rows than the target asks for, this helper keeps
    all available issue rows and fills the rest of the row budget with good rows.
    """
    # Sort first so the returned training set still reads in time order.
    work = frame.sort_values("Time UTC").reset_index(drop=True).copy()
    if rows_limit is None or len(work) <= rows_limit:
        return work
    if not 0 < balanced_issue_share < 1:
        raise ValueError("balanced_issue_share must be in the open interval (0, 1)")

    # Split the reviewed rows into good vs issue groups, then sample each group to the requested share.
    normalized_good_labels = [int(label) for label in good_labels]
    normalized_issue_labels = [int(label) for label in issue_labels]
    numeric_target = pd.to_numeric(work[target_column], errors="coerce")
    good_frame = work.loc[numeric_target.isin(normalized_good_labels)].reset_index(drop=True)
    issue_frame = work.loc[numeric_target.isin(normalized_issue_labels)].reset_index(drop=True)
    dedupe_columns = [column for column in ["source_file", "Time UTC"] if column in work.columns]

    def drop_selected_duplicates(candidate_frame: pd.DataFrame) -> pd.DataFrame:
        if dedupe_columns:
            return candidate_frame.drop_duplicates(subset=dedupe_columns)
        return candidate_frame.drop_duplicates()

    def remove_already_selected(candidate_frame: pd.DataFrame, selected_frame: pd.DataFrame) -> pd.DataFrame:
        if selected_frame.empty:
            return candidate_frame
        if dedupe_columns:
            selected_keys = set(map(tuple, selected_frame[dedupe_columns].to_numpy()))
            keep_mask = ~candidate_frame[dedupe_columns].apply(tuple, axis=1).isin(selected_keys)
            return candidate_frame.loc[keep_mask]
        return candidate_frame.drop(selected_frame.index, errors="ignore")

    requested_issue_rows = min(max(int(round(rows_limit * balanced_issue_share)), 0), int(rows_limit))
    issue_target = min(requested_issue_rows, len(issue_frame))
    good_target = int(rows_limit) - issue_target

    # Keep as many issue rows as the target allows. If issues are scarce, keep all of them.
    issue_sample = evenly_spaced_take(issue_frame, issue_target)
    good_sample = evenly_spaced_take(good_frame, good_target)

    sampled_frame = pd.concat([good_sample, issue_sample], ignore_index=True)
    sampled_frame = drop_selected_duplicates(sampled_frame).sort_values("Time UTC").reset_index(drop=True)

    # If one group ran out of rows, refill from rows that have not already been selected.
    if len(sampled_frame) < rows_limit:
        fill_needed = int(rows_limit) - len(sampled_frame)
        fill_candidates = remove_already_selected(work, sampled_frame).reset_index(drop=True)
        fill_frame = evenly_spaced_take(fill_candidates, fill_needed)
        sampled_frame = pd.concat([sampled_frame, fill_frame], ignore_index=True)
        sampled_frame = drop_selected_duplicates(sampled_frame).sort_values("Time UTC").reset_index(drop=True)

    return sampled_frame.iloc[: int(rows_limit)].sort_values("Time UTC").reset_index(drop=True)


def subsample_train_frame(
    frame: pd.DataFrame,
    *,
    rows_limit: int | None,
    strategy: str,
    target_column: str,
    good_labels: list[int] | tuple[int, ...],
    issue_labels: list[int] | tuple[int, ...],
    issue_rows: int,
    balanced_issue_share: float,
) -> pd.DataFrame:
    """Apply the requested train-only subsampling rule and return the result.

    Validation and test stay unchanged; only the training split is reduced here.
    """
    # Sort once up front so every strategy starts from the same time-ordered training frame.
    ordered = frame.sort_values("Time UTC").reset_index(drop=True).copy()
    # "full_train" means keep the complete training split without further shrinking.
    if rows_limit is None or len(ordered) <= rows_limit or strategy == "full_train":
        return ordered
    # Otherwise dispatch to the selected subsampling rule.
    if strategy == "time_spread":
        return subsample_time_spread(ordered, rows_limit)
    if strategy == "issue_focused":
        return subsample_issue_focused(
            ordered,
            rows_limit=rows_limit,
            target_column=target_column,
            issue_labels=issue_labels,
            issue_rows=issue_rows,
        )
    if strategy == "balanced_reviewed":
        return subsample_balanced_reviewed(
            ordered,
            rows_limit=rows_limit,
            target_column=target_column,
            good_labels=good_labels,
            issue_labels=issue_labels,
            balanced_issue_share=balanced_issue_share,
        )
    raise ValueError(f"Unsupported train subset strategy: {strategy}")


In [ ]:
# Load only the columns needed for label accounting and split design.
overview_columns = ["Time UTC", "source_file", TARGET_FLAG]
dataset_label_df = load_full_row_level_frame(selected_paths, columns=overview_columns)
dataset_label_df[TARGET_FLAG] = pd.to_numeric(dataset_label_df[TARGET_FLAG], errors="coerce")
if "source_file" in dataset_label_df.columns:
    dataset_label_df["source_file"] = dataset_label_df["source_file"].astype("category")

print(
    {
        "selected_parts": len(selected_paths),
        "selected_rows": len(dataset_label_df),
        "overview_columns_loaded": overview_columns,
    }
)


In [ ]:
reviewed_model_accounting = show_reviewed_model_row_accounting(
    dataset_label_df,
    target_flag=TARGET_FLAG,
    task_mode=TASK_MODE,
    good_labels=GOOD_LABELS,
    issue_labels=ISSUE_LABELS,
    model_row_limit=REVIEWED_MODEL_ROW_LIMIT,  # Explicitly computed in the visible row-budget cell above.
    data_fraction=BUDGET_DATA_FRACTION,
    flag_palette=DEFAULT_FLAG_PALETTE,
    label_meanings=FLAG_EXAMPLE_LABEL_MEANINGS,
)

# reviewed_label_df keeps every usable reviewed row.
# reviewed_model_df applies DATA_FRACTION and is the dataframe used for split comparison below.
reviewed_label_df = reviewed_model_accounting["reviewed_label_df"]
reviewed_model_df = reviewed_model_accounting["reviewed_model_df"]
active_labels = reviewed_model_accounting["active_labels"]
model_good_labels = reviewed_model_accounting["model_good_labels"]
model_issue_labels = reviewed_model_accounting["model_issue_labels"]
selected_target_counts = reviewed_model_accounting["selected_target_counts"]
selected_reviewed_counts = reviewed_model_accounting["selected_reviewed_counts"]
reviewed_model_counts = reviewed_model_accounting["reviewed_model_counts"]
row_accounting_summary = reviewed_model_accounting["row_summary"]
REVIEWED_MODEL_ROW_LIMIT = reviewed_model_accounting["summary"]["reviewed_model_row_limit"]


#### Verify How The Flags Change Across Time

Before choosing a split strategy, look at **where the reviewed modelling labels live on the time axis**.

This is an important dataset check because:

- issue labels may cluster into only a few time periods,
- the label mix may shift across time,
- and those patterns directly affect how a split behaves.

These plots are not model outputs. They are a dataset sanity check before we define train / validation / test.

One plot shows the reviewed modelling label mix through time. Another removes the good reviewed rows so you can focus only on how the issue labels are distributed.


In [ ]:
temporal_summary_bundle = show_temporal_flag_summary(
    reviewed_model_df,
    target_flag=TARGET_FLAG,
    selected_path_count=len(selected_paths),
    temporal_summary_bin_count=TEMPORAL_SUMMARY_BIN_COUNT,
    good_labels=GOOD_LABELS,
    issue_labels=ISSUE_LABELS,
    flag_palette=DEFAULT_FLAG_PALETTE,
    target_display_name=FLAG_EXAMPLE_DISPLAY_NAME,
)
temporal_bin_count = temporal_summary_bundle["bin_count"]
temporal_labels = temporal_summary_bundle["labels"]
temporal_counts = temporal_summary_bundle["counts"]
temporal_shares = temporal_summary_bundle["shares"]
temporal_issue_only_shares = temporal_summary_bundle["issue_only_shares"]
temporal_summary = temporal_summary_bundle["summary"]

print(
    {
        "temporal_bin_count": temporal_bin_count,
        "time_bins_shown": len(temporal_summary),
        "issue_labels": ISSUE_LABELS,
        "max_issue_share_pct": float(temporal_summary["issue_share_pct"].max()) if len(temporal_summary) else 0.0,
        "min_issue_share_pct": float(temporal_summary["issue_share_pct"].min()) if len(temporal_summary) else 0.0,
    }
)


#### Split Strategy Controls

These settings control the split comparison in the next cells.

- `global_contiguous`: one early / middle / late cut across the whole time axis.
- `per_source_contiguous`: make an early / middle / late cut inside each source file, then combine the pieces.
- `interleaved_blocks`: keep short local time blocks intact, but distribute those blocks across train / validation / test.

`INTERLEAVED_BLOCK_ROWS` controls how large each local block is in the interleaved strategy.

Validation and test stay unbalanced so they reflect the natural distribution produced by the split.


In [ ]:
SUPPORTED_SPLIT_STRATEGIES = (
    "global_contiguous",
    "per_source_contiguous",
    "interleaved_blocks",
)
INTERLEAVED_BLOCK_ROWS = 1024

show_setup_json(
    {
        "SUPPORTED_SPLIT_STRATEGIES": SUPPORTED_SPLIT_STRATEGIES,
        "INTERLEAVED_BLOCK_ROWS": INTERLEAVED_BLOCK_ROWS,
        "TRAIN_FRACTION": TRAIN_FRACTION,
        "VALIDATION_FRACTION": VALIDATION_FRACTION,
    }
)


#### Compare Split Strategies

This comparison uses the **full reviewed rows** from the selected parquet parts.

That is deliberate: we want the split decision to be based on the real reviewed dataset, not on a tiny notebook sample.

The table below keeps the comparison simple:

- rows per split,
- issue rows and issue share per split,
- and the time span covered by each split.


In [ ]:
comparison_labels = [
    label
    for label in sorted(set(model_good_labels).union(model_issue_labels))
    if label in set(reviewed_model_df["model_target"].dropna().astype(int).unique().tolist())
]
present_issue_labels = [label for label in model_issue_labels if label in comparison_labels]

split_strategy_labels = {
    "global_contiguous": "Global contiguous",
    "per_source_contiguous": "Per-source contiguous",
    "interleaved_blocks": "Interleaved blocks",
}

strategy_frames_by_name = {}
strategy_comparison_rows = []

for strategy_name in SUPPORTED_SPLIT_STRATEGIES:
    strategy_frames = split_reviewed_frame(
        reviewed_model_df,
        train_fraction=TRAIN_FRACTION,
        validation_fraction=VALIDATION_FRACTION,
        strategy=strategy_name,
        block_rows=INTERLEAVED_BLOCK_ROWS,
    )
    strategy_frames_by_name[strategy_name] = strategy_frames

    for split_name, split_frame in strategy_frames.items():
        split_labels = pd.to_numeric(split_frame["model_target"], errors="coerce").dropna().astype(int)
        label_counts = split_labels.value_counts().sort_index()
        issue_rows = int(label_counts.reindex(present_issue_labels, fill_value=0).sum())
        strategy_comparison_rows.append(
            {
                "strategy": split_strategy_labels[strategy_name],
                "split": split_name,
                "rows": len(split_frame),
                "issue_rows": issue_rows,
                "issue_share_pct": round(100 * issue_rows / max(len(split_frame), 1), 2),
                "time_start": split_frame["Time UTC"].min().isoformat() if len(split_frame) else None,
                "time_end": split_frame["Time UTC"].max().isoformat() if len(split_frame) else None,
            }
        )

strategy_comparison_df = pd.DataFrame(strategy_comparison_rows)
display(
    strategy_comparison_df.style.format(
        {
            "rows": "{:,.0f}",
            "issue_rows": "{:,.0f}",
            "issue_share_pct": "{:.2f}",
        }
    )
)


**Visual comparison**

Now that you have seen the labels through time, we can ask a more focused question: **how does the split itself change the balance?**

We compare the same three strategies on the same reviewed modelling dataset:

- **Global contiguous**: earliest rows in train, middle rows in validation, latest rows in test.
- **Per-source contiguous**: each source file contributes an early / middle / late slice.
- **Interleaved blocks**: short local time blocks stay intact, but the blocks rotate across splits.

The timeline plot shows where each split lands across the full selected time period. The composition plot then shows the reviewed modelling label mix for each strategy and, separately, the issue-label mix without the good rows.


In [ ]:
strategy_timeline_bundle = show_split_strategy_timeline(
    strategy_frames_by_name,
    strategy_display_names=split_strategy_labels,
    figure_title="Where each split strategy places train / validation / test over time",
)
strategy_timeline_summary = strategy_timeline_bundle["summary"]

strategy_plot_bundle = show_split_strategy_comparison(
    strategy_frames_by_name,
    strategy_display_names=split_strategy_labels,
    issue_labels=present_issue_labels,
    label_column="model_target",
    flag_palette=DEFAULT_FLAG_PALETTE,
    legend_title="model target",
    figure_title="How the split strategy changes label balance on the reviewed modelling dataset",
)
strategy_shares_by_name = strategy_plot_bundle["shares"]
strategy_issue_only_shares_by_name = strategy_plot_bundle["issue_only_shares"]

show_setup_json(
    {
        "DATA_FRACTION": DATA_FRACTION,
        "REVIEWED_MODEL_ROW_LIMIT": REVIEWED_MODEL_ROW_LIMIT,
        "reviewed_model_rows": len(reviewed_model_df),
        "INTERLEAVED_BLOCK_ROWS": INTERLEAVED_BLOCK_ROWS,
        "present_issue_labels": present_issue_labels,
    }
)


**Questions to ask before you lock the split**

Before choosing a fixed split, pause and look for structure that a simple label-balance table cannot show on its own:

1. Are the issue flags isolated points, or do they cluster into longer episodes?
2. Could one real bad-data event be getting cut across train, validation, and test?
3. If the model uses windows or lagged context, would nearby timestamps leak information across split boundaries?
4. Does this split answer the question you care about: generalize to nearby conditions, or generalize to a completely new bad episode?

Keep those questions in mind as you pick the split strategy below. The advanced notebook comes back to them with a stronger episode-aware fixed split.


#### Choose And Build The Fixed Split

This is the single fixed train / validation / test split used by the model sections that follow.

The next cell keeps the editable choices visible. The utility helper handles the repetitive work: loading reviewed labels, creating the modelling target, applying the split, and returning the standard dataframe names used later.


In [ ]:
# FIXED_SPLIT_STRATEGY controls how reviewed rows are assigned to train / validation / test.
# "global_contiguous": split the whole time-ordered dataset into one early/middle/late split.
# "per_source_contiguous": split each source file separately, then combine the matching split pieces.
# "interleaved_blocks": alternate fixed-size time blocks across train / validation / test.
FIXED_SPLIT_STRATEGY = "global_contiguous"
if FIXED_SPLIT_STRATEGY not in SUPPORTED_SPLIT_STRATEGIES:
    raise ValueError(
        f"Unsupported split strategy: {FIXED_SPLIT_STRATEGY}. Choose from {SUPPORTED_SPLIT_STRATEGIES}."
    )

# REVIEWED_MODEL_ROW_LIMIT was computed from DATA_FRACTION and the usable reviewed row count above.
# With DATA_FRACTION = 1.0, this equals the full usable reviewed count.

# FIXED_SPLIT_BLOCK_ROWS only changes interleaved splits.
# The contiguous strategies ignore block rows because they split by continuous time ranges.
FIXED_SPLIT_BLOCK_ROWS = INTERLEAVED_BLOCK_ROWS if FIXED_SPLIT_STRATEGY == "interleaved_blocks" else None

show_setup_json(
    {
        "DATA_FRACTION": DATA_FRACTION,
        "REVIEWED_MODEL_ROW_LIMIT": REVIEWED_MODEL_ROW_LIMIT,
        "FIXED_SPLIT_STRATEGY": FIXED_SPLIT_STRATEGY,
        "FIXED_SPLIT_BLOCK_ROWS": FIXED_SPLIT_BLOCK_ROWS,
    }
)


In [ ]:
# Build the reviewed modelling table and fixed split in one utility call.
reviewed_split_bundle = build_reviewed_modelling_split(
    # selected_paths: row-level parquet parts selected from the active cache earlier in Part 3.
    selected_paths=selected_paths,
    # target_flag: the reviewed QC/custom label column we want to learn from.
    target_flag=TARGET_FLAG,
    # task_mode: "multiclass" keeps label values; "binary" converts to issue vs not-issue.
    task_mode=TASK_MODE,
    # good_labels / issue_labels: define which reviewed labels become usable modelling rows.
    good_labels=GOOD_LABELS,
    issue_labels=ISSUE_LABELS,
    # model_row_limit: optional row cap derived from DATA_FRACTION for faster live runs.
    model_row_limit=REVIEWED_MODEL_ROW_LIMIT,
    # train_fraction / validation_fraction: test receives the remaining fraction.
    train_fraction=TRAIN_FRACTION,
    validation_fraction=VALIDATION_FRACTION,
    # split_strategy / split_block_rows: the fixed split choices from the control cell above.
    split_strategy=FIXED_SPLIT_STRATEGY,
    split_block_rows=FIXED_SPLIT_BLOCK_ROWS,
    # measurement_columns: kept with the returned metadata for later model sections.
    measurement_columns=MEASUREMENT_COLUMNS,
)

# Restore the standard dataframe names used by the ML sections:
# reviewed_model_df, train_full_df, valid_df, test_df, model_good_labels, model_issue_labels, etc.
globals().update(reviewed_split_bundle["notebook_values"])
show_reviewed_modelling_split_build(reviewed_split_bundle)


#### Review The Fixed Split

The review helper shows the split sizes, issue coverage, time span, full label mix, and issue-only label mix. Validation and test are natural outputs of the split; they are not rebalanced.


In [ ]:
fixed_split_review = show_fixed_split_review(
    # split_frames: the exact train / validation / test dataframes all model sections reuse.
    {"train": train_full_df, "validation": valid_df, "test": test_df},
    # issue_labels: which model_target values count as issue classes in the issue-only plot.
    issue_labels=model_issue_labels,
    # split_strategy_label: human-readable title for the plots and tables.
    split_strategy_label=split_strategy_labels.get(FIXED_SPLIT_STRATEGY, FIXED_SPLIT_STRATEGY),
    # flag_palette: consistent colours for label values across the notebook.
    flag_palette=DEFAULT_FLAG_PALETTE,
)

# Keep the displayed review objects available in case later cells or participants want to inspect them.
split_overview = fixed_split_review["overview"]
split_counts = fixed_split_review["counts"]
split_shares = fixed_split_review["shares"]
split_issue_only_shares = fixed_split_review["issue_only_shares"]


### Phase 2 — Design The Train-Only Subset

The reviewed modelling dataset is fixed now. From this point on, validation and test stay unchanged. The next decisions only affect **which training rows** we hand to the models.


#### Training Subset Comparison Settings

The fixed split is fixed now. `DATA_FRACTION` controls the train-only row budget used in the next block, while `TRAIN_SUBSET_STRATEGY` later chooses *which* training rows are kept.

`BALANCED_REVIEWED_TARGET_ISSUE_SHARE` only matters for `balanced_reviewed`. It is a target, not a guarantee: if the full training split does not contain enough issue rows, the helper keeps all available issue rows and fills the remaining row budget with good rows.

Validation and test stay untouched so the comparison stays fair.


In [ ]:
SUPPORTED_TRAIN_SUBSET_STRATEGIES = ("full_train", "time_spread", "balanced_reviewed", "issue_focused")
# TRAIN_SUBSET_MAX_ROWS and ISSUE_ROWS_PER_FILE were set in the visible row-budget cell above.
# Set TRAIN_SUBSET_MAX_ROWS = None here if you want this section to use the full train split.
BALANCED_REVIEWED_TARGET_ISSUE_SHARE = 0.25
if not 0 < BALANCED_REVIEWED_TARGET_ISSUE_SHARE < 1:
    raise ValueError("BALANCED_REVIEWED_TARGET_ISSUE_SHARE must be in the interval (0, 1).")

show_setup_json(
    {
        "DATA_FRACTION": DATA_FRACTION,
        "TRAIN_SUBSET_MAX_ROWS": TRAIN_SUBSET_MAX_ROWS,
        "ISSUE_ROWS_PER_FILE": ISSUE_ROWS_PER_FILE,
        "BALANCED_REVIEWED_TARGET_ISSUE_SHARE": BALANCED_REVIEWED_TARGET_ISSUE_SHARE,
    }
)


#### Compare Train-Only Subset Strategies

These next cells stay focused on the training set only. Validation and test do not change here.

First we summarise the candidate train-only subsets. Then we visualise how the target mix changes inside train.


In [ ]:
train_subset_preview_source = train_full_df.copy()
train_subset_rows_limit = (
    len(train_full_df)
    if TRAIN_SUBSET_MAX_ROWS is None
    else min(int(TRAIN_SUBSET_MAX_ROWS), len(train_full_df))
)

available_train_issue_rows = int(train_subset_preview_source["model_target"].isin(model_issue_labels).sum())
max_possible_balanced_issue_share = min(available_train_issue_rows, train_subset_rows_limit) / max(train_subset_rows_limit, 1)

train_subset_preview_frames = {"full_train": train_subset_preview_source}
for subset_strategy in ("time_spread", "balanced_reviewed", "issue_focused"):
    train_subset_preview_frames[subset_strategy] = subsample_train_frame(
        train_subset_preview_source,
        rows_limit=train_subset_rows_limit,
        strategy=subset_strategy,
        target_column="model_target",
        good_labels=model_good_labels,
        issue_labels=model_issue_labels,
        issue_rows=ISSUE_ROWS_PER_FILE,
        balanced_issue_share=BALANCED_REVIEWED_TARGET_ISSUE_SHARE,
    )

train_subset_labels = {
    "full_train": "full train",
    "time_spread": "time-spread subset",
    "balanced_reviewed": (
        f"balanced-reviewed subset ({BALANCED_REVIEWED_TARGET_ISSUE_SHARE:.0%} target; "
        f"max {max_possible_balanced_issue_share:.1%})"
    ),
    "issue_focused": "issue-focused subset",
}

train_subset_overview_rows = []
for subset_name, subset_frame in train_subset_preview_frames.items():
    subset_counts = subset_frame["model_target"].value_counts().sort_index()
    issue_rows = int(subset_counts.reindex(model_issue_labels, fill_value=0).sum())
    train_subset_overview_rows.append(
        {
            "subset": train_subset_labels[subset_name],
            "rows": len(subset_frame),
            "issue_rows": issue_rows,
            "issue_share_pct": round(100 * issue_rows / max(len(subset_frame), 1), 2),
            "target_issue_share_pct": (
                round(100 * BALANCED_REVIEWED_TARGET_ISSUE_SHARE, 2)
                if subset_name == "balanced_reviewed"
                else None
            ),
            "max_possible_issue_share_pct": round(
                100 * min(available_train_issue_rows, len(subset_frame)) / max(len(subset_frame), 1),
                2,
            ),
            "time_start": subset_frame["Time UTC"].min().isoformat(),
            "time_end": subset_frame["Time UTC"].max().isoformat(),
        }
    )
train_subset_overview = pd.DataFrame(train_subset_overview_rows).set_index("subset")

train_budget_counts = pd.DataFrame(
    {
        subset_name: subset_frame["model_target"].value_counts().sort_index()
        for subset_name, subset_frame in train_subset_preview_frames.items()
    }
).fillna(0).astype(int)
train_budget_shares = train_budget_counts.div(train_budget_counts.sum(axis=0), axis=1).fillna(0.0)
display(
    train_subset_overview.style.format(
        {
            "rows": "{:,.0f}",
            "issue_rows": "{:,.0f}",
            "issue_share_pct": "{:.2f}",
            "target_issue_share_pct": "{:.2f}",
            "max_possible_issue_share_pct": "{:.2f}",
        },
        na_rep="-",
    )
)


In [ ]:
train_budget_plot = train_budget_shares.T.copy()
train_budget_plot.index = [
    f"{train_subset_labels[subset_name]} (n={int(train_budget_counts[subset_name].sum()):,})"
    for subset_name in train_budget_plot.index
]
palette = [DEFAULT_FLAG_PALETTE.get(int(label), "#64748b") for label in train_budget_plot.columns]

fig, ax = plt.subplots(figsize=(11.8, 4.6))
train_budget_plot.plot(kind="barh", stacked=True, ax=ax, color=palette, width=0.6)
ax.set_xlim(0, 1)
ax.xaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_xlabel("Share of reviewed rows in train")
ax.set_ylabel("")
ax.set_title("How the train-only subset strategy changes the training set")
ax.grid(axis="x", alpha=0.2)
ax.legend(title="target_label", bbox_to_anchor=(1.01, 1.0), loc="upper left")
plt.tight_layout()
plt.show()


#### Choose The Train-Only Subset

Pick the train-only subset strategy you want the models to fit on. This is the main knob to change when you want to see how train-set construction affects the models.

Validation and test stay fixed. This choice only affects the training rows that are handed to the models.


In [ ]:
# Change this directly to compare how different train-only subset choices affect the models.
# Options: "full_train", "time_spread", "balanced_reviewed", "issue_focused"
TRAIN_SUBSET_STRATEGY = "time_spread"
if TRAIN_SUBSET_STRATEGY not in SUPPORTED_TRAIN_SUBSET_STRATEGIES:
    raise ValueError(
        f"Unsupported TRAIN_SUBSET_STRATEGY: {TRAIN_SUBSET_STRATEGY}. Choose from {SUPPORTED_TRAIN_SUBSET_STRATEGIES}."
    )

train_df = subsample_train_frame(
    train_full_df,
    rows_limit=TRAIN_SUBSET_MAX_ROWS,
    strategy=TRAIN_SUBSET_STRATEGY,
    target_column="model_target",
    good_labels=model_good_labels,
    issue_labels=model_issue_labels,
    issue_rows=ISSUE_ROWS_PER_FILE,
    balanced_issue_share=BALANCED_REVIEWED_TARGET_ISSUE_SHARE,
)

show_setup_json(
    {
        "TRAIN_SUBSET_STRATEGY": TRAIN_SUBSET_STRATEGY,
        "TRAIN_SUBSET_MAX_ROWS": TRAIN_SUBSET_MAX_ROWS,
        "BALANCED_REVIEWED_TARGET_ISSUE_SHARE": BALANCED_REVIEWED_TARGET_ISSUE_SHARE,
        "train_rows": len(train_df),
        "validation_rows": len(valid_df),
        "test_rows": len(test_df),
    }
)


## Part 4 — Random Forest

We start with a strong tabular baseline: one row becomes one supervised example, and the forest learns to separate QC classes from engineered numeric features.


#### Skip Ahead To This ML Section

Use the next cell if you want to jump straight to this ML section, or if you already ran earlier sections but your kernel state was cleared. It restores the cache paths, dataset-specific label names/meanings, reviewed modelling rows, train/validation/test split, and train-only subset.

Uncomment the cell, adjust only the inputs you care about, and run it before continuing.


In [ ]:
# # Uncomment and run this cell to jump straight to this ML section.
# from pathlib import Path
# import sys
#
# NOTEBOOK_ROOT = Path.cwd()
# for candidate_root in [NOTEBOOK_ROOT, *NOTEBOOK_ROOT.parents]:
#     if (candidate_root / "notebooks").exists() and (candidate_root / "scripts").exists():
#         NOTEBOOK_ROOT = candidate_root
#         break
# if str(NOTEBOOK_ROOT) not in sys.path:
#     sys.path.insert(0, str(NOTEBOOK_ROOT))
#
# from scripts.session1_resume_utils import load_ml_section_state
#
# globals().update(
#     load_ml_section_state(
#         notebook_root=NOTEBOOK_ROOT,  # Repo root; usually leave this alone.
#         dataset_profile_id="ctd_conductivity",  # Dataset preset: "ctd_conductivity", "fluorometer_turbidity", "oxygen", or "conductivity_plugs".
#         data_fraction=0.1,  # Dataset size: 0.1 is a quick 10% run, 0.9 is a larger 90% run, and 1.0 removes row caps.
#         split_strategy="global_contiguous",  # Split style: "global_contiguous", "per_source_contiguous", or "interleaved_blocks".
#         train_subset_strategy="time_spread",  # Train-only subset: "time_spread", "issue_focused", or "balanced_reviewed".
#     )
# )


### 🌲 Supervised Learning: Random Forest

A **Random Forest** is an ensemble of many decision trees.

Here is the basic idea:

1. make many slightly different training samples from the data,
2. train one decision tree on each sample,
3. let the trees vote on the final class.

Each tree asks simple if/then questions such as:

- is conductivity above some threshold?
- did temperature change sharply?
- is the measurement happening at a particular time of day?

In this notebook, the Random Forest does **not** train on the raw reviewed modelling table directly.
Right before fitting the forest, we turn each reviewed row into a tabular feature vector that includes:

- the selected measurement columns themselves,
- `abs_delta` features that measure how sharply each measurement changed from the previous row,
- and simple clock features such as hour, minute, and day of year.

That feature-building step is specific to the Random Forest baseline. The CNN and transformer sections later do **not** use this same tabular feature table.

Why this is a good Session 1 model:

- it works well on numeric tabular data,
- it usually needs less feature scaling ceremony than many other models,
- it gives us interpretable outputs such as feature importance.

Limits to keep in mind:

- it does not naturally understand long sequential context,
- it only learns from the features we explicitly give it,
- rare classes can still be difficult.


![Random Forest bagging illustration](../assets/session1/random_forest_bagging_illustration.png)

Diagram idea: bootstrap samples are drawn from the original dataset, separate trees are trained, and their predictions are aggregated into one model decision.

Image credit: Harry585, Wikimedia Commons, CC BY-SA 4.0. Source: [File:Random Forest Bagging Illustration.png](https://commons.wikimedia.org/wiki/File:Random_Forest_Bagging_Illustration.png)


### Random Forest Settings

These settings control the baseline forest we train below.

Main variables:

- `n_estimators`: number of trees in the forest.
- `max_depth`: maximum depth of each tree. `None` means grow until other stopping rules apply.
- `min_samples_leaf`: minimum number of samples allowed in a leaf.
- `min_samples_split`: minimum number of samples needed to split an internal node.
- `max_features`: how many features each split is allowed to consider. Common choices are `"sqrt"`, `"log2"`, or `None`.
- `class_weight`: how strongly to compensate for imbalance. Common choices are `None`, `"balanced"`, and `"balanced_subsample"`.
- `trees_per_step`: how many trees to add each time we print progress.

Forests do **not** train epoch-by-epoch like neural networks. To make progress visible, we grow the forest in chunks using `warm_start=True` and print the validation F1 after each chunk.

One practical note: when we use `warm_start=True`, this notebook converts `"balanced"` or `"balanced_subsample"` into one fixed class-weight dictionary computed from `y_train`. That avoids a scikit-learn warning and keeps the weighting consistent across growth steps.


In [ ]:
RF_CONFIG = {
    "imputer_strategy": "median",
    "n_estimators": 200,
    "trees_per_step": 25,
    "max_depth": None,
    "min_samples_leaf": 2,
    "min_samples_split": 2,
    "max_features": "sqrt",
    "class_weight": "balanced_subsample",
    "verbose": 0,
}

display(pd.Series(RF_CONFIG, name="value").rename_axis("rf_parameter").to_frame())


#### Random Forest Row Budget

The Random Forest uses the same `DATA_FRACTION` budget as the rest of the notebook. The next cell applies the derived caps and prints the actual row counts used for train, validation, and test.


In [ ]:
# These row caps were set in the visible row-budget cell above.
# Set any of them to None here if you want the random forest section to use the full split.

RF_TRAIN_CAP_STRATEGY = (
    TRAIN_SUBSET_STRATEGY
    if TRAIN_SUBSET_STRATEGY != "full_train"
    else "time_spread"
)

rf_train_fit_df = (
    sample_frame_by_strategy(
        train_df,
        rows_limit=RF_TRAIN_MAX_ROWS,
        sample_strategy=RF_TRAIN_CAP_STRATEGY,
        target_flag=TARGET_FLAG,
        good_labels=GOOD_LABELS,
        issue_labels=ISSUE_LABELS,
        issue_rows=ISSUE_ROWS_PER_FILE,
        balanced_issue_share=BALANCED_REVIEWED_TARGET_ISSUE_SHARE,
    )
    if RF_TRAIN_MAX_ROWS is not None and len(train_df) > RF_TRAIN_MAX_ROWS
    else train_df.copy()
)
rf_valid_eval_df = (
    evenly_spaced_take(valid_df.sort_values("Time UTC").reset_index(drop=True), RF_VALIDATION_MAX_ROWS)
    if RF_VALIDATION_MAX_ROWS is not None and len(valid_df) > RF_VALIDATION_MAX_ROWS
    else valid_df.copy()
)
rf_test_eval_df = (
    evenly_spaced_take(test_df.sort_values("Time UTC").reset_index(drop=True), RF_TEST_MAX_ROWS)
    if RF_TEST_MAX_ROWS is not None and len(test_df) > RF_TEST_MAX_ROWS
    else test_df.copy()
)

show_setup_json(
    {
        "DATA_FRACTION": DATA_FRACTION,
        "RF_TRAIN_MAX_ROWS": RF_TRAIN_MAX_ROWS,
        "RF_VALIDATION_MAX_ROWS": RF_VALIDATION_MAX_ROWS,
        "RF_TEST_MAX_ROWS": RF_TEST_MAX_ROWS,
        "rf_train_cap_strategy": RF_TRAIN_CAP_STRATEGY,
        "rf_train_rows_used": len(rf_train_fit_df),
        "rf_validation_rows_used": len(rf_valid_eval_df),
        "rf_test_rows_used": len(rf_test_eval_df),
    }
)


In [ ]:
import gc

rf_split_rows = materialize_reviewed_split_frames(
    selected_paths,
    {
        "train": rf_train_fit_df,
        "validation": rf_valid_eval_df,
        "test": rf_test_eval_df,
    },
    columns=ROW_USE_COLUMNS,
    target_flag=TARGET_FLAG,
    task_mode=task_mode,
    good_labels=GOOD_LABELS,
    issue_labels=ISSUE_LABELS,
)
rf_train_rows_df = rf_split_rows["train"]
rf_valid_rows_df = rf_split_rows["validation"]
rf_test_rows_df = rf_split_rows["test"]

# Build the Random Forest feature table here. This is the step where the shared
# helper adds the tabular baseline features:
# - the selected measurement columns,
# - per-column `abs_delta` change features,
# - and simple UTC clock features.
rf_train_fit_df, feature_columns, _ = build_model_frame(
    rf_train_rows_df,
    target_flag=TARGET_FLAG,
    measurement_columns=measurement_columns,
    task_mode=task_mode,
    good_labels=GOOD_LABELS,
    issue_labels=ISSUE_LABELS,
    model_row_limit=None,
)
rf_valid_eval_df, _, _ = build_model_frame(
    rf_valid_rows_df,
    target_flag=TARGET_FLAG,
    measurement_columns=measurement_columns,
    task_mode=task_mode,
    good_labels=GOOD_LABELS,
    issue_labels=ISSUE_LABELS,
    model_row_limit=None,
)
rf_test_eval_df, _, _ = build_model_frame(
    rf_test_rows_df,
    target_flag=TARGET_FLAG,
    measurement_columns=measurement_columns,
    task_mode=task_mode,
    good_labels=GOOD_LABELS,
    issue_labels=ISSUE_LABELS,
    model_row_limit=None,
)

# The raw materialized split frames are no longer needed once the feature tables exist.
# Releasing them keeps high-fraction runs from carrying duplicate copies of the data.
for _rf_name in ["rf_split_rows", "rf_train_rows_df", "rf_valid_rows_df", "rf_test_rows_df"]:
    globals().pop(_rf_name, None)
gc.collect()

# Step 1: fit the imputer once on training data and reuse it for validation/test.
rf_imputer = SimpleImputer(strategy=RF_CONFIG.get("imputer_strategy", "median"))
X_train_rf = rf_imputer.fit_transform(rf_train_fit_df[feature_columns]).astype(np.float32, copy=False)
X_valid_rf = rf_imputer.transform(rf_valid_eval_df[feature_columns]).astype(np.float32, copy=False)
y_train_rf = rf_train_fit_df["model_target"].reset_index(drop=True)
y_valid_rf = rf_valid_eval_df["model_target"].reset_index(drop=True)
y_test_rf = rf_test_eval_df["model_target"].reset_index(drop=True)

# Step 2: compute one stable class-weight dictionary for the whole training split.
requested_class_weight = RF_CONFIG.get("class_weight")
if requested_class_weight in {"balanced", "balanced_subsample"}:
    rf_classes = np.array(sorted(pd.Series(y_train_rf).dropna().unique()))
    rf_weight_values = compute_class_weight(
        class_weight="balanced",
        classes=rf_classes,
        y=y_train_rf,
    )
    effective_class_weight = {
        int(label) if isinstance(label, (np.integer, int)) else label: float(weight)
        for label, weight in zip(rf_classes.tolist(), rf_weight_values.tolist())
    }
else:
    effective_class_weight = requested_class_weight

print(
    {
        "requested_class_weight": requested_class_weight,
        "effective_class_weight": effective_class_weight,
    }
)

# Step 3: build the forest itself. We use warm_start so we can grow it in chunks and print progress.
rf_model = RandomForestClassifier(
    n_estimators=RF_CONFIG["trees_per_step"],
    warm_start=True,
    max_depth=RF_CONFIG["max_depth"],
    min_samples_leaf=RF_CONFIG["min_samples_leaf"],
    min_samples_split=RF_CONFIG["min_samples_split"],
    max_features=RF_CONFIG["max_features"],
    class_weight=effective_class_weight,
    n_jobs=-1,
    random_state=SEED,
    verbose=RF_CONFIG["verbose"],
)

rf_progress_rows = []
total_trees = RF_CONFIG["n_estimators"]
trees_per_step = RF_CONFIG["trees_per_step"]
growth_schedule = list(range(trees_per_step, total_trees + 1, trees_per_step))
if not growth_schedule:
    growth_schedule = [total_trees]
elif growth_schedule[-1] != total_trees:
    growth_schedule.append(total_trees)

for tree_count in growth_schedule:
    rf_model.set_params(n_estimators=tree_count)
    rf_model.fit(X_train_rf, y_train_rf)
    current_valid_predictions = rf_model.predict(X_valid_rf)
    current_valid_f1 = f1_score(
        y_valid_rf,
        current_valid_predictions,
        average=report_average(task_mode),
        zero_division=0,
    )
    rf_progress_rows.append({"trees_built": tree_count, "validation_f1": float(current_valid_f1)})
    print(f"Built {tree_count:>3} trees | validation F1 = {current_valid_f1:.4f}")

# Step 4: wrap the fitted pieces into a reusable sklearn pipeline object.
rf_pipeline = Pipeline(
    steps=[
        ("imputer", rf_imputer),
        ("model", rf_model),
    ]
)

X_test_rf = rf_imputer.transform(rf_test_eval_df[feature_columns]).astype(np.float32, copy=False)
valid_predictions = rf_model.predict(X_valid_rf)
test_predictions = rf_model.predict(X_test_rf)
labels = sorted(pd.unique(pd.concat([y_train_rf, y_valid_rf, y_test_rf])))

display(pd.DataFrame(rf_progress_rows))

with RF_MODEL_PATH.open("wb") as handle:
    pickle.dump(
        {
            "pipeline": rf_pipeline,
            "feature_columns": feature_columns,
            "labels": labels,
            "task_mode": task_mode,
        },
        handle,
    )

print(
    {
        "saved_random_forest_model": str(RF_MODEL_PATH),
        "rf_train_rows_used": int(len(rf_train_fit_df)),
        "rf_validation_rows_used": int(len(rf_valid_eval_df)),
        "rf_test_rows_used": int(len(rf_test_eval_df)),
    }
)

# The fitted pipeline keeps the trained model and imputer. The arrays and
# temporary fit frames below are only needed while fitting and predicting.
for _rf_name in [
    "X_train_rf",
    "X_valid_rf",
    "X_test_rf",
    "current_valid_predictions",
    "rf_train_fit_df",
    "rf_valid_eval_df",
    "rf_model",
    "rf_imputer",
    "y_train_rf",
]:
    globals().pop(_rf_name, None)
gc.collect()


In [ ]:
# Report validation and test metrics separately for the RF evaluation slices.
metric_rows = []
for split_name, y_true, y_pred in [
    ("validation", y_valid_rf, valid_predictions),
    ("test", y_test_rf, test_predictions),
]:
    split_f1 = f1_score(y_true, y_pred, average=report_average(task_mode), zero_division=0)
    metric_rows.append({"split": split_name, "f1": round(float(split_f1), 4)})
    print(f"{split_name.title()} macro/binary F1: {split_f1:.4f}")
    print(classification_report(y_true, y_pred, labels=labels, zero_division=0))

display(pd.DataFrame(metric_rows))

# Plot normalised confusion matrices for both validation and test.
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
ConfusionMatrixDisplay.from_predictions(
    y_valid_rf,
    valid_predictions,
    labels=labels,
    display_labels=[str(label) for label in labels],
    normalize="true",
    cmap="Blues",
    values_format=".2f",
    xticks_rotation=45,
    ax=axes[0],
)
axes[0].set_title(f"Validation confusion matrix for {target_name}")

ConfusionMatrixDisplay.from_predictions(
    y_test_rf,
    test_predictions,
    labels=labels,
    display_labels=[str(label) for label in labels],
    normalize="true",
    cmap="Blues",
    values_format=".2f",
    xticks_rotation=45,
    ax=axes[1],
)
axes[1].set_title(f"Test confusion matrix for {target_name}")
fig.suptitle("How to read these plots: each row is a true class, and darker diagonal cells are better.", y=1.03)
plt.tight_layout()
plt.show()


In [ ]:
# Feature importance tells us which columns the forest used most strongly.
feature_importances = pd.Series(
    rf_pipeline.named_steps["model"].feature_importances_,
    index=feature_columns,
).sort_values(ascending=False)

top_importances = feature_importances.head(12).sort_values()
top_importances.plot(kind="barh", figsize=(8, 6), title="Top Random Forest feature importances")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()
plt.close()
display(feature_importances.head(12).rename("importance").to_frame())

# Show a few test-set mistakes to ground the discussion in real timestamps.
error_preview_columns = [
    "Time UTC",
    TARGET_FLAG,
    "issue",
    PLOT_MEASUREMENT_COLUMN,
    PLOT_SECONDARY_COLUMN,
]
error_preview_columns.extend(
    [
        column
        for column in MEASUREMENT_COLUMNS
        if column not in error_preview_columns
    ][:1]
)
error_preview_columns = [column for column in error_preview_columns if column in rf_test_eval_df.columns]
error_frame = rf_test_eval_df[error_preview_columns].copy()
error_frame["prediction"] = test_predictions
error_frame["correct"] = error_frame["prediction"] == y_test_rf.to_numpy()
error_examples = error_frame.loc[~error_frame["correct"]].head(12)
print({"test_errors_shown": len(error_examples), "test_error_rate": round(float((~error_frame["correct"]).mean()), 4)})
display(error_examples)

# The test prediction arrays are no longer needed after the mistake preview.
for _rf_name in [
    "rf_test_eval_df",
    "y_test_rf",
    "test_predictions",
    "error_frame",
    "error_examples",
    "feature_importances",
    "top_importances",
]:
    globals().pop(_rf_name, None)
gc.collect()


### Date-Range Demo: Predict QC Flags with the Random Forest

The test-set metrics above summarise performance across the whole held-out split. This mini-workflow asks a more concrete question:

**What does the Random Forest predict on one specific interval of time?**

Use UTC strings such as `"2025-09-10 00:00:00Z"` if you want to override the default range.


In [ ]:
RF_RANGE_START = None
RF_RANGE_END = None
RF_AUTO_SELECT_TEST_RANGE = True
RF_MAX_POINTS_TO_PLOT = 800

print(
    {
        "RF_RANGE_START": RF_RANGE_START,
        "RF_RANGE_END": RF_RANGE_END,
        "RF_AUTO_SELECT_TEST_RANGE": RF_AUTO_SELECT_TEST_RANGE,
        "RF_MAX_POINTS_TO_PLOT": RF_MAX_POINTS_TO_PLOT,
    }
)


In [ ]:
import gc

rf_range_selection = select_time_range(
    test_df,
    time_column="Time UTC",
    label_column=TARGET_FLAG,
    start=RF_RANGE_START,
    end=RF_RANGE_END,
    auto_select=RF_AUTO_SELECT_TEST_RANGE,
    max_points=RF_MAX_POINTS_TO_PLOT,
)

rf_interval_rows = load_rows_for_time_range(
    metadata,
    row_cache_dir=Path(ROW_CACHE_DIR),
    start=rf_range_selection["start"],
    end=rf_range_selection["end"],
    columns=ROW_USE_COLUMNS,
)

if rf_interval_rows.empty:
    print("No row-level data was found in the requested Random Forest range.")
else:
    rf_interval_model_df, _, _ = build_model_frame(
        rf_interval_rows,
        target_flag=TARGET_FLAG,
        measurement_columns=measurement_columns,
        task_mode=task_mode,
        model_row_limit=None,
    )
    rf_interval_model_df = rf_interval_model_df[
        (rf_interval_model_df["Time UTC"] >= rf_range_selection["start"])
        & (rf_interval_model_df["Time UTC"] <= rf_range_selection["end"])
    ].reset_index(drop=True)

    if rf_interval_model_df.empty:
        print("The selected Random Forest range did not contain usable labelled rows after preparation.")
    else:
        rf_interval_predictions = rf_pipeline.predict(rf_interval_model_df[feature_columns])
        rf_interval_origin = infer_interval_origin(
            rf_range_selection["start"],
            rf_range_selection["end"],
            {"train": train_df, "validation": valid_df, "test": test_df},
        )
        rf_interval_metrics = compute_interval_classification_metrics(
            rf_interval_model_df["model_target"],
            rf_interval_predictions,
            labels=labels,
            average=report_average(task_mode),
            target_names=[str(label) for label in labels],
        )
        rf_plot_palette = DEFAULT_FLAG_PALETTE if task_mode == "multiclass" else {0: "#1f77b4", 1: "#d62728"}
        rf_true_intervals = build_labeled_intervals(
            rf_interval_model_df,
            time_column="Time UTC",
            label_column="model_target",
        )
        rf_pred_frame = rf_interval_model_df[["Time UTC"]].copy()
        rf_pred_frame["predicted_label"] = rf_interval_predictions
        rf_pred_intervals = build_labeled_intervals(
            rf_pred_frame,
            time_column="Time UTC",
            label_column="predicted_label",
        )

        print(
            {
                "selection_mode": rf_range_selection["selection_mode"],
                "selected_priority_flag": rf_range_selection["selected_label"],
                "interval_origin": rf_interval_origin,
                "range_start": rf_range_selection["start"].isoformat(),
                "range_end": rf_range_selection["end"].isoformat(),
                "rows_in_interval": int(len(rf_interval_model_df)),
                "interval_f1": rf_interval_metrics["f1"],
            }
        )
        print(rf_interval_metrics["report_text"])

        display(
            pd.DataFrame(
                {
                    "true_count": pd.Series(rf_interval_model_df["model_target"]).value_counts().sort_index(),
                    "predicted_count": pd.Series(rf_interval_predictions).value_counts().sort_index(),
                }
            ).fillna(0).astype(int)
        )

        rf_demo_figure = plot_time_series_with_bands(
            rf_interval_model_df,
            band_specs=[
                {"title": f"True {FLAG_EXAMPLE_DISPLAY_NAME}", "intervals": rf_true_intervals, "palette": rf_plot_palette},
                {"title": f"RF {FLAG_EXAMPLE_DISPLAY_NAME} predictions", "intervals": rf_pred_intervals, "palette": rf_plot_palette},
            ],
            measurement_column=PLOT_MEASUREMENT_COLUMN,
            secondary_column=PLOT_SECONDARY_COLUMN,
            max_points=RF_MAX_POINTS_TO_PLOT,
            title="Random Forest predictions on a selected time range",
            label_meanings=FLAG_EXAMPLE_LABEL_MEANINGS,
            legend_title=FLAG_EXAMPLE_DISPLAY_NAME,
        )
        plt.show()
        plt.close(rf_demo_figure)

        rf_cm_fig, rf_cm_ax = plt.subplots(figsize=(5, 4))
        ConfusionMatrixDisplay(
            confusion_matrix=rf_interval_metrics["confusion_matrix"],
            display_labels=rf_interval_metrics["display_labels"],
        ).plot(ax=rf_cm_ax, cmap="Blues", colorbar=False)
        rf_cm_ax.set_title("Random Forest confusion matrix on the selected range")
        plt.tight_layout()
        plt.show()
        plt.close(rf_cm_fig)

# Random Forest interval intermediates can be rebuilt by rerunning this cell.
for _rf_name in [
    "rf_interval_rows",
    "rf_interval_model_df",
    "rf_interval_predictions",
    "rf_interval_metrics",
    "rf_true_intervals",
    "rf_pred_frame",
    "rf_pred_intervals",
]:
    globals().pop(_rf_name, None)
gc.collect()


### Reflection Questions: Random Forest

1. Which input features seem most responsible for the mistakes in this interval: raw measurements, change features, or time-of-day features?
2. Do the errors line up with the class imbalance we saw earlier, or do they suggest a different modelling problem?
3. If you wanted to improve this interval specifically, would you change the features, the target, or the forest settings first?

Add your own notes or follow-up questions here:

- 
- 
- 


## Part 5 — k-means

Next we switch to an unsupervised lens: instead of predicting flags directly, we group windows with similar summary behaviour and interpret those clusters.


#### Skip Ahead To This ML Section

Use the next cell if you want to jump straight to this ML section, or if you already ran earlier sections but your kernel state was cleared. It restores the cache paths, dataset-specific label names/meanings, reviewed modelling rows, train/validation/test split, and train-only subset.

Uncomment the cell, adjust only the inputs you care about, and run it before continuing.


In [ ]:
# # Uncomment and run this cell to jump straight to this ML section.
# from pathlib import Path
# import sys
#
# NOTEBOOK_ROOT = Path.cwd()
# for candidate_root in [NOTEBOOK_ROOT, *NOTEBOOK_ROOT.parents]:
#     if (candidate_root / "notebooks").exists() and (candidate_root / "scripts").exists():
#         NOTEBOOK_ROOT = candidate_root
#         break
# if str(NOTEBOOK_ROOT) not in sys.path:
#     sys.path.insert(0, str(NOTEBOOK_ROOT))
#
# from scripts.session1_resume_utils import load_ml_section_state
#
# globals().update(
#     load_ml_section_state(
#         notebook_root=NOTEBOOK_ROOT,  # Repo root; usually leave this alone.
#         dataset_profile_id="ctd_conductivity",  # Dataset preset: "ctd_conductivity", "fluorometer_turbidity", "oxygen", or "conductivity_plugs".
#         data_fraction=0.1,  # Dataset size: 0.1 is a quick 10% run, 0.9 is a larger 90% run, and 1.0 removes row caps.
#         split_strategy="global_contiguous",  # Split style: "global_contiguous", "per_source_contiguous", or "interleaved_blocks".
#         train_subset_strategy="time_spread",  # Train-only subset: "time_spread", "issue_focused", or "balanced_reviewed".
#     )
# )


### 🧭 Unsupervised Learning: k-means On Window Features

Supervised learning uses labels. Unsupervised learning does **not**.

Here we use **k-means**, which groups rows or windows into clusters based on similarity in feature space. The cluster IDs themselves do not carry meaning ahead of time. We interpret them *after* fitting by looking at:

- how many examples ended up in each cluster,
- the mean issue rate inside each cluster,
- where the cluster sits in feature space,
- and what the underlying time-series examples look like.

This is useful when you want to surface interesting operating regimes or suspicious periods even before a label model is mature.

One subtle point: the cluster numbers and colours are arbitrary. They only become meaningful after we interpret them using the plots and summary statistics below.


### 🔎 k-means Settings

Main variables:

- `n_clusters`: how many clusters we ask the algorithm to find.
- `n_init`: how many random initializations to try. `"auto"` is a good default in current scikit-learn.
- `KMEANS_FEATURE_MODE`: inherited from the dataset profile. Some datasets cluster best on window summaries, while spike-driven ones may cluster better on row-level features.

The number of cached windows loaded for clustering is derived from `DATA_FRACTION`. If you want to experiment with k-means itself, the most useful first change is usually `n_clusters`.


In [ ]:
KMEANS_CONFIG = {
    "n_clusters": 5,
    "n_init": "auto",
}
# KMEANS_WINDOW_LIMIT was set in the visible row-budget cell above.
# Set KMEANS_WINDOW_LIMIT = None here if you want to cluster every selected window/row.
KMEANS_EXAMPLES_PER_CLUSTER = 1
KMEANS_EXAMPLE_CONTEXT_POINTS = 7500
KMEANS_EXAMPLE_HIGHLIGHT_ALPHA = 0.22

display(pd.Series(KMEANS_CONFIG, name="value").rename_axis("kmeans_parameter").to_frame())
print(
    {
        "DATA_FRACTION": DATA_FRACTION,
        "KMEANS_FEATURE_MODE": KMEANS_FEATURE_MODE,
        "KMEANS_WINDOW_LIMIT": KMEANS_WINDOW_LIMIT,
        "KMEANS_EXAMPLES_PER_CLUSTER": KMEANS_EXAMPLES_PER_CLUSTER,
        "KMEANS_EXAMPLE_CONTEXT_POINTS": KMEANS_EXAMPLE_CONTEXT_POINTS,
        "KMEANS_EXAMPLE_HIGHLIGHT_ALPHA": KMEANS_EXAMPLE_HIGHLIGHT_ALPHA,
    }
)


In [ ]:
if KMEANS_FEATURE_MODE == "window_summary":
    window_df = pd.read_parquet(window_cache_path, columns=WINDOW_USE_COLUMNS)
    window_df["window_start"] = pd.to_datetime(window_df["window_start"], utc=True)
    window_df["window_end"] = pd.to_datetime(window_df["window_end"], utc=True)
    window_df = (
        window_df[window_df["source_file"].isin(selected_source_files)]
        .sort_values("window_start")
        .reset_index(drop=True)
    )
    window_df = evenly_spaced_take(window_df, KMEANS_WINDOW_LIMIT, time_column="window_start")
    cluster_source_df = window_df
    kmeans_feature_columns = [
        column
        for column in cluster_source_df.columns
        if column.endswith("_mean") or column.endswith("_std")
    ]
    cluster_x_column = f"{PLOT_MEASUREMENT_COLUMN}_mean"
    cluster_y_column = f"{PLOT_SECONDARY_COLUMN}_mean"
    cluster_axis_label_suffix = "mean"
else:
    # Row-level k-means should use the same visible row budget as window-summary k-means.
    # Without this sample, row-level feature mode would materialise the full reviewed dataset.
    cluster_selection_df = evenly_spaced_take(
        reviewed_model_df[["Time UTC", "source_file"]].copy(),
        KMEANS_WINDOW_LIMIT,
        time_column="Time UTC",
    )
    cluster_source_rows = load_selected_row_level_frame(
        selected_paths,
        cluster_selection_df,
        columns=ROW_USE_COLUMNS,
    )
    cluster_source_df, feature_columns, _ = build_model_frame(
        cluster_source_rows,
        target_flag=TARGET_FLAG,
        measurement_columns=measurement_columns,
        task_mode=task_mode,
        good_labels=GOOD_LABELS,
        issue_labels=ISSUE_LABELS,
        model_row_limit=None,
    )
    cluster_source_df["issue"] = cluster_source_df["issue"].astype(int)
    cluster_source_df["window_start"] = pd.to_datetime(cluster_source_df["Time UTC"], utc=True)
    cluster_source_df["window_end"] = pd.to_datetime(cluster_source_df["Time UTC"], utc=True)
    cluster_source_df["issue_rate"] = cluster_source_df["issue"].astype(float)
    kmeans_feature_columns = [column for column in feature_columns if column in cluster_source_df.columns]
    cluster_x_column = PLOT_MEASUREMENT_COLUMN
    cluster_y_column = PLOT_SECONDARY_COLUMN
    cluster_axis_label_suffix = "value"

if cluster_x_column not in cluster_source_df.columns or cluster_y_column not in cluster_source_df.columns:
    raise ValueError(
        f"k-means plotting columns are missing for profile {DATASET_PROFILE_ID}: "
        f"{cluster_x_column!r}, {cluster_y_column!r}"
    )

window_df, cluster_summary = fit_kmeans(
    cluster_source_df,
    n_clusters=KMEANS_CONFIG["n_clusters"],
    seed=SEED,
    n_init=KMEANS_CONFIG["n_init"],
    feature_mode=KMEANS_FEATURE_MODE,
    feature_columns=kmeans_feature_columns,
)
display(cluster_summary.round({"mean_issue_rate": 3, "max_issue_rate": 3, "avg_distance": 3}))


In [ ]:
# Plot a readable sample of clustered points so the scatter does not become an unreadable blob.
plot_window_df = evenly_spaced_take(window_df, 5000, time_column="window_start")
cluster_ids = sorted(cluster_summary.index.tolist())
cluster_colors = plt.cm.tab10(np.linspace(0, 1, len(cluster_ids)))
cluster_palette = {cluster_id: cluster_colors[idx] for idx, cluster_id in enumerate(cluster_ids)}

fig, axes = plt.subplots(1, 2, figsize=(17, 6))
cluster_centers = (
    plot_window_df.groupby("cluster")[[cluster_x_column, cluster_y_column]]
    .mean()
    .reindex(cluster_ids)
)

legend_handles = []
for cluster_id in cluster_ids:
    subset = plot_window_df[plot_window_df["cluster"] == cluster_id]
    axes[0].scatter(
        subset[cluster_x_column],
        subset[cluster_y_column],
        s=18,
        alpha=0.55,
        color=cluster_palette[cluster_id],
        edgecolors="none",
    )
    axes[0].scatter(
        cluster_centers.loc[cluster_id, cluster_x_column],
        cluster_centers.loc[cluster_id, cluster_y_column],
        s=220,
        marker="*",
        color=cluster_palette[cluster_id],
        edgecolors="black",
        linewidths=0.8,
    )
    legend_handles.append(
        Line2D(
            [0],
            [0],
            marker="o",
            color="w",
            markerfacecolor=cluster_palette[cluster_id],
            markersize=8,
            label=(
                f"Cluster {cluster_id} | n={int(cluster_summary.loc[cluster_id, 'window_count'])} | "
                f"mean issue={cluster_summary.loc[cluster_id, 'mean_issue_rate']:.2f}"
            ),
        )
    )

axes[0].set_title(f"k-means clusters in {KMEANS_FEATURE_MODE.replace('_', ' ')} feature space")
axes[0].set_xlabel(f"{PLOT_MEASUREMENT_COLUMN} ({cluster_axis_label_suffix})")
axes[0].set_ylabel(f"{PLOT_SECONDARY_COLUMN} ({cluster_axis_label_suffix})")
axes[0].grid(alpha=0.25)
axes[0].legend(handles=legend_handles, title="Cluster legend", loc="upper left", bbox_to_anchor=(1.02, 1.0), frameon=True)

bar_positions = np.arange(len(cluster_ids))
bar_values = [cluster_summary.loc[cluster_id, "mean_issue_rate"] for cluster_id in cluster_ids]
bar_colors = [cluster_palette[cluster_id] for cluster_id in cluster_ids]
axes[1].bar(bar_positions, bar_values, color=bar_colors, edgecolor="black", linewidth=0.6)
axes[1].set_xticks(bar_positions)
axes[1].set_xticklabels([f"Cluster {cluster_id}" for cluster_id in cluster_ids], rotation=30)
axes[1].set_ylim(0, 1.05)
axes[1].set_ylabel("Mean issue rate")
axes[1].set_title("Average issue rate by cluster")
for idx, cluster_id in enumerate(cluster_ids):
    axes[1].text(
        idx,
        bar_values[idx] + 0.02,
        f"n={int(cluster_summary.loc[cluster_id, 'window_count'])}\n{bar_values[idx]:.1%}",
        ha="center",
        va="bottom",
        fontsize=9,
    )

plt.tight_layout()
plt.show()

interesting_windows = window_df.sort_values(
    ["issue_rate", "distance_to_centroid"],
    ascending=[False, False],
).head(10)
interesting_windows = interesting_windows[
    ["window_start", "window_end", "source_file", "cluster", "issue_rate", "distance_to_centroid"]
].copy()
interesting_windows["source_file"] = interesting_windows["source_file"].map(clean_source_file_label)
display(interesting_windows.round({"issue_rate": 3, "distance_to_centroid": 3}))


### Looking At Real Time-Series Windows From Each Cluster

The scatter plot shows where clusters sit in feature space, but it does not show what the underlying sensor behaviour actually looked like.

The next plot closes that loop. For each cluster, we pick a representative window that sits close to its centroid, then:

- show a wider time-series context from the original row-level parquet,
- show the true target-label regions from that underlying data,
- highlight the exact window used in k-means,
- mark the datapoints inside that highlighted window.

This makes it much easier to interpret clusters as real operating regimes rather than abstract colored dots.


In [ ]:
cluster_example_figure, cluster_example_windows = plot_cluster_window_examples(
    window_df,
    source_to_row_part=source_to_row_part,
    measurement_column=PLOT_MEASUREMENT_COLUMN,
    secondary_column=PLOT_SECONDARY_COLUMN,
    target_flag=TARGET_FLAG,
    good_labels=GOOD_LABELS,
    examples_per_cluster=KMEANS_EXAMPLES_PER_CLUSTER,
    context_points=KMEANS_EXAMPLE_CONTEXT_POINTS,
    highlight_alpha=KMEANS_EXAMPLE_HIGHLIGHT_ALPHA,
    flag_palette=DEFAULT_FLAG_PALETTE,
    target_display_name=FLAG_EXAMPLE_DISPLAY_NAME,
    label_meanings=FLAG_EXAMPLE_LABEL_MEANINGS,
)
plt.show()
display(cluster_example_windows)


### Date-Range Demo: See Which Clusters Appear in a Specific Interval

k-means does not predict target labels directly. Instead, it groups short windows with similar summary behaviour.

This demo lets us ask: **what kinds of windows show up inside one selected time range, and how do their issue rates differ?**


In [ ]:
KMEANS_RANGE_START = "2026-02-9T12:56:27.707000+00:00"
KMEANS_RANGE_END = "2026-02-11T12:56:27.707000+00:00"
KMEANS_AUTO_SELECT_TEST_RANGE = True
KMEANS_MAX_POINTS_TO_PLOT = 800

print(
    {
        "KMEANS_RANGE_START": KMEANS_RANGE_START,
        "KMEANS_RANGE_END": KMEANS_RANGE_END,
        "KMEANS_AUTO_SELECT_TEST_RANGE": KMEANS_AUTO_SELECT_TEST_RANGE,
        "KMEANS_MAX_POINTS_TO_PLOT": KMEANS_MAX_POINTS_TO_PLOT,
    }
)


In [ ]:
kmeans_range_selection = select_time_range(
    test_df,
    time_column="Time UTC",
    label_column=TARGET_FLAG,
    start=KMEANS_RANGE_START,
    end=KMEANS_RANGE_END,
    auto_select=KMEANS_AUTO_SELECT_TEST_RANGE,
    max_points=KMEANS_MAX_POINTS_TO_PLOT,
)

kmeans_interval_rows = load_rows_for_time_range(
    metadata,
    row_cache_dir=Path(ROW_CACHE_DIR),
    start=kmeans_range_selection["start"],
    end=kmeans_range_selection["end"],
    columns=ROW_USE_COLUMNS,
)
kmeans_interval_windows = window_df[
    (window_df["window_start"] <= kmeans_range_selection["end"])
    & (window_df["window_end"] >= kmeans_range_selection["start"])
].copy()

if kmeans_interval_rows.empty or kmeans_interval_windows.empty:
    print("No overlapping k-means windows were found in the requested range.")
else:
    kmeans_interval_origin = infer_interval_origin(
        kmeans_range_selection["start"],
        kmeans_range_selection["end"],
        {"train": train_df, "validation": valid_df, "test": test_df},
    )
    kmeans_interval_intervals = merge_adjacent_intervals(
        kmeans_interval_windows.rename(
            columns={"window_start": "start", "window_end": "end", "cluster": "label"}
        )[["start", "end", "label"]]
    )
    kmeans_cluster_counts = (
        kmeans_interval_windows.groupby("cluster")
        .agg(
            window_count=("cluster", "size"),
            mean_issue_rate=("issue_rate", "mean"),
            max_issue_rate=("issue_rate", "max"),
        )
        .sort_index()
    )

    print(
        {
            "selection_mode": kmeans_range_selection["selection_mode"],
            "selected_priority_flag": kmeans_range_selection["selected_label"],
            "interval_origin": kmeans_interval_origin,
            "range_start": kmeans_range_selection["start"].isoformat(),
            "range_end": kmeans_range_selection["end"].isoformat(),
            "row_points_in_interval": int(len(kmeans_interval_rows)),
            "window_count_in_interval": int(len(kmeans_interval_windows)),
        }
    )
    display(kmeans_cluster_counts.round(3))

    kmeans_demo_figure = plot_time_series_with_bands(
        kmeans_interval_rows,
        band_specs=[
            {
                "title": "k-means clusters",
                "intervals": kmeans_interval_intervals,
                "palette": cluster_palette,
            }
        ],
        measurement_column=PLOT_MEASUREMENT_COLUMN,
        secondary_column=PLOT_SECONDARY_COLUMN,
        max_points=KMEANS_MAX_POINTS_TO_PLOT,
        title="k-means cluster assignments on a selected time range",
        legend_title="cluster",
    )
    plt.show()


### Reflection Questions: k-means

1. Do these cluster spans look like real operating regimes, or do some clusters still seem hard to interpret physically?
2. If you changed `n_clusters`, which regions in this interval do you expect would split apart or merge together?
3. How closely do the cluster spans line up with target-label changes, and what does that say about the usefulness of unsupervised learning here?

Add your own notes or follow-up questions here:

- 
- 
- 


## Part 6 — 1D CNN

Now we keep short stretches of signal together as one training example and see how a sequence model behaves when it learns local time patterns directly.


#### Skip Ahead To This ML Section

Use the next cell if you want to jump straight to this ML section, or if you already ran earlier sections but your kernel state was cleared. It restores the cache paths, dataset-specific label names/meanings, reviewed modelling rows, train/validation/test split, and train-only subset.

Uncomment the cell, adjust only the inputs you care about, and run it before continuing.


In [ ]:
# # Uncomment and run this cell to jump straight to this ML section.
# from pathlib import Path
# import sys
#
# NOTEBOOK_ROOT = Path.cwd()
# for candidate_root in [NOTEBOOK_ROOT, *NOTEBOOK_ROOT.parents]:
#     if (candidate_root / "notebooks").exists() and (candidate_root / "scripts").exists():
#         NOTEBOOK_ROOT = candidate_root
#         break
# if str(NOTEBOOK_ROOT) not in sys.path:
#     sys.path.insert(0, str(NOTEBOOK_ROOT))
#
# from scripts.session1_resume_utils import load_ml_section_state
#
# globals().update(
#     load_ml_section_state(
#         notebook_root=NOTEBOOK_ROOT,  # Repo root; usually leave this alone.
#         dataset_profile_id="ctd_conductivity",  # Dataset preset: "ctd_conductivity", "fluorometer_turbidity", "oxygen", or "conductivity_plugs".
#         data_fraction=0.1,  # Dataset size: 0.1 is a quick 10% run, 0.9 is a larger 90% run, and 1.0 removes row caps.
#         split_strategy="global_contiguous",  # Split style: "global_contiguous", "per_source_contiguous", or "interleaved_blocks".
#         train_subset_strategy="time_spread",  # Train-only subset: "time_spread", "issue_focused", or "balanced_reviewed".
#     )
# )


### Sequential Model: 1D CNN

A **convolutional neural network (CNN)** applies small learnable filters across an ordered signal. For images that order is two-dimensional. In this notebook, the order is **time**.

The key idea is:

1. a short filter looks at a local pattern,
2. the same filter slides across the whole sequence,
3. later layers combine those detected patterns into a final prediction.

For this scalar QC task, a 1D CNN can learn patterns such as:

- sudden spikes or drops,
- flat segments,
- repeated local shapes across several sensor channels.

Why this is useful here:

- Random Forest treats each row mostly as a tabular snapshot,
- CNN keeps a short stretch of signal together as one example,
- the training loop also gives us a chance to teach batching, validation checkpoints, and early stopping.

The CNN section runs by default in this notebook and follows standard training practice:

- train / validation / test split,
- class-aware loss weighting,
- best-checkpoint saving based on validation F1,
- early stopping,
- mini-batch training through `DataLoader`,
- pinned-memory and multi-worker loading when the local machine supports it.

Useful references: [PyTorch tutorial on defining neural networks](https://docs.pytorch.org/tutorials/beginner/blitz/neural_networks_tutorial), [PyTorch DataLoader docs](https://pytorch.org/docs/stable/data.html), [PyTorch performance tuning guide](https://pytorch.org/tutorials/recipes/recipes/tuning_guide.html)


![Convolutional network diagram](../assets/session1/convolutional_network.png)

Diagram idea: small filters slide across the input, produce feature maps, and later layers combine those maps into a final prediction.

In this notebook the same logic is applied to **1D time windows** rather than 2D images.

Image credit: Aphex34, Wikimedia Commons, CC BY-SA 4.0. Source: [File:Convolutional Network.png](https://commons.wikimedia.org/wiki/File:Convolutional_Network.png)


### 🎛️ CNN Settings

These settings control the baseline 1D CNN and its training loop.

Model-shape variables:

- `window_size`: number of time steps in each training window.
- `output_mode`: choose `"window"` for one label per window or `"per_timestep"` for one label at every time step.
- `conv_channels`: number of filters in each convolution layer.
- `dropout`: dropout probability used for regularisation.
- `label_reduction`: how row-level labels become one window label. This only matters when `output_mode="window"`.

Optimisation variables:

- `epochs`: maximum number of passes through the training data.
- `batch_size`: number of windows per optimisation step.
- `learning_rate`: optimiser step size.
- `weight_decay`: L2-style regularisation for the optimiser.
- `patience` and `min_delta`: early-stopping controls.
- `lr_decay_factor` and `lr_patience`: learning-rate scheduler controls.
- `gradient_clip_norm`: cap on gradient size to stabilize training.

DataLoader variables:

- `num_workers`: worker processes for loading batches.
- `pin_memory`: can speed up host-to-GPU transfer.
- `persistent_workers`: keeps workers alive between epochs.
- `prefetch_factor`: how many batches each worker prepares ahead of time.

Dataset-aware default:

- the turbidity and conductivity-plug profiles start in `"per_timestep"` mode because short local events are easy to lose inside mixed windows,
- the other profiles start in `"window"` mode to keep the first baseline simpler.


In [ ]:
# CNN_CONFIG controls the learning problem, model size, and training loop.
# Start with these defaults, then change one or two values at a time so results stay interpretable.
CNN_CONFIG = {
    # Number of consecutive timestamps given to the CNN for one example.
    "window_size": 128,
    # "window" predicts one label for the whole window; "per_timestep" predicts one label per row.
    "output_mode": DEFAULT_SEQUENCE_OUTPUT_MODE,
    "epochs": 10,
    "batch_size": 128,
    "learning_rate": 1e-3,
    "weight_decay": 1e-4,
    "patience": 3,
    "min_delta": 0.001,
    "dropout": 0.2,
    "lr_decay_factor": 0.5,
    "lr_patience": 1,
    "gradient_clip_norm": 1.0,
    # Each number is the number of filters in a Conv1D layer. More filters can learn richer patterns but cost more memory.
    "conv_channels": [32, 64],
    "label_reduction": "worst",
}

# DataLoader settings control how batches are moved into PyTorch.
# num_workers=0 is slower but safest inside shared notebook environments.
_loader_torch = globals().get("torch", None)
CNN_LOADER_CONFIG = {
    "num_workers": 0,
    "pin_memory": bool(_loader_torch is not None and _loader_torch.cuda.is_available()),
    "persistent_workers": False,
    "prefetch_factor": 2,
}

display(pd.Series(CNN_CONFIG, name="value").rename_axis("cnn_parameter").to_frame())
display(pd.Series(CNN_LOADER_CONFIG, name="value").rename_axis("loader_parameter").to_frame())


### Utility For Window Labels

The CNN and transformer sections now support two output styles:

- `"window"`: one prediction for the whole window,
- `"per_timestep"`: one prediction for every point inside the window.

When you choose `"window"`, we need one short rule for turning several row-level target labels inside a window into one label. The helper below keeps that reduction in one place so the later cells stay easier to read.


In [ ]:
# Reduce row-level labels to one window-level label for sequence models.
def reduce_window_target(
    values: np.ndarray,
    mode: str,
    severity_order: tuple[int, ...] = (1, 3, 4, 9),
) -> int:
    labels = [int(value) for value in values if pd.notna(value)]
    if not labels:
        return severity_order[0]
    effective_order = tuple(sorted(set(severity_order).union(labels)))
    severity_rank = {label: index for index, label in enumerate(effective_order)}
    if mode == "worst":
        return max(labels, key=lambda label: severity_rank.get(label, -1))
    counts = pd.Series(labels).value_counts()
    tied_labels = counts[counts == counts.max()].index.tolist()
    return max(tied_labels, key=lambda label: severity_rank.get(int(label), -1))


### CNN Step 1: Turn Rows into Windows

A CNN expects a fixed-size tensor, not an arbitrary-length table.

In this step we:

- collect the measurement columns we want to model,
- group consecutive rows into fixed windows,
- either reduce many row-level target labels into one window label or keep one target per timestamp,
- split the windows with the active train / validation / test strategy,
- normalise each sensor channel using only the training split.

One important detail: the baseline CNN does **not** take the full Random Forest feature table as input. It sees windows of the selected measurement columns only. That means it gets more temporal context than the RF, but less hand-engineered context.

This is a good place to pause and ask: what information do we lose when we compress several row labels into one window label, and when is that simplification actually helpful?


In [ ]:
# Run this cell when you are ready to work through the CNN section.
# Earlier notebook parts do not need PyTorch, so we import it here.
try:
    import torch
    from torch import nn
    from torch.utils.data import DataLoader, TensorDataset

    PYTORCH_READY = True
    CNN_READY = True
    print({
        "torch": torch.__version__,
        "cuda_available": torch.cuda.is_available(),
        "device_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    })
except ImportError as exc:
    PYTORCH_READY = False
    CNN_READY = False
    print("PyTorch is not installed in this environment.")
    raise SystemExit(exc)


In [ ]:
if not CNN_READY:
    CNN_RUN = False
    print("CNN training cell skipped.")
else:
    # Load the row-level measurements for the already chosen train/validation/test split.
    # This keeps the CNN evaluation aligned with the Random Forest fixed split.
    sequence_split_frames_materialized = materialize_reviewed_split_frames(
        selected_paths,
        {"train": train_df, "validation": valid_df, "test": test_df},
        columns=ROW_USE_COLUMNS,
        target_flag=TARGET_FLAG,
        task_mode=task_mode,
        good_labels=GOOD_LABELS,
        issue_labels=ISSUE_LABELS,
    )
    # Pull the settings into short variable names because they are used throughout the CNN cells.
    WINDOW_SIZE = CNN_CONFIG["window_size"]
    CNN_OUTPUT_MODE = CNN_CONFIG["output_mode"]
    # Convert rows into fixed-length windows.
    # The helper returns X arrays shaped like (windows, time, channels) and y labels for each split.
    cnn_bundle = build_sequence_split_bundle_from_frames(
        sequence_split_frames_materialized,
        measurement_columns=measurement_columns,
        target_column="model_target",
        task_mode=task_mode,
        output_mode=CNN_OUTPUT_MODE,
        window_size=WINDOW_SIZE,
        label_reduction=CNN_CONFIG["label_reduction"],
    )
    # class_labels maps model output indexes back to the original QC labels.
    class_labels = cnn_bundle.class_labels

    # A classifier needs at least two classes and at least one complete window in every split.
    if task_mode == "multiclass" and len(class_labels) < 2:
        CNN_RUN = False
        print(
            {
                "output_mode": CNN_OUTPUT_MODE,
                "windows_total": int(len(cnn_bundle.X_train) + len(cnn_bundle.X_valid) + len(cnn_bundle.X_test)),
                "active_labels": class_labels,
                "skip_reason": "Need at least two active classes to train the CNN in multiclass mode.",
            }
        )
    elif len(cnn_bundle.X_train) == 0 or len(cnn_bundle.X_valid) == 0 or len(cnn_bundle.X_test) == 0:
        CNN_RUN = False
        print(
            {
                "output_mode": CNN_OUTPUT_MODE,
                "split_strategy": FIXED_SPLIT_STRATEGY,
                "train_subset_strategy": TRAIN_SUBSET_STRATEGY,
                "skip_reason": "At least one split produced no full windows for this configuration.",
            }
        )
    else:
        # Move to channel-first layout expected by Conv1D.
        X_train = np.transpose(cnn_bundle.X_train, (0, 2, 1))
        X_valid = np.transpose(cnn_bundle.X_valid, (0, 2, 1))
        X_test = np.transpose(cnn_bundle.X_test, (0, 2, 1))
        # Keep the labels beside the transposed feature arrays; labels are not normalised.
        y_train = cnn_bundle.y_train
        y_valid = cnn_bundle.y_valid
        y_test = cnn_bundle.y_test

        # Normalise each sensor channel using only the training split.
        # Validation and test use the training mean/std so evaluation does not peek at held-out data.
        channel_mean = X_train.mean(axis=(0, 2), keepdims=True)
        channel_std = X_train.std(axis=(0, 2), keepdims=True) + 1e-6
        X_train = (X_train - channel_mean) / channel_std
        X_valid = (X_valid - channel_mean) / channel_std
        X_test = (X_test - channel_mean) / channel_std

        CNN_RUN = True
        print(
            {
                "output_mode": CNN_OUTPUT_MODE,
                "split_strategy": FIXED_SPLIT_STRATEGY,
                "train_subset_strategy": TRAIN_SUBSET_STRATEGY,
                "fixed_split_block_rows": FIXED_SPLIT_BLOCK_ROWS if FIXED_SPLIT_STRATEGY in {"interleaved_blocks", "episode_aware"} else None,
                "windows_total": int(len(X_train) + len(X_valid) + len(X_test)),
                "train_windows": int(len(X_train)),
                "validation_windows": int(len(X_valid)),
                "test_windows": int(len(X_test)),
                "window_size": int(WINDOW_SIZE),
                "channels": int(len(measurement_columns)),
            }
        )


### CNN Step 2: Build the Model and the DataLoaders

Here we create four pieces:

- the neural network itself,
- the loss function,
- the optimiser and learning-rate scheduler,
- the `DataLoader` objects that stream mini-batches during training.

The output toggle changes the final prediction head:

- in `"window"` mode, the CNN pools across time and emits one label for the whole window,
- in `"per_timestep"` mode, the CNN keeps the time axis and emits one label per point.

Notice that the `DataLoader` keeps the full dataset in CPU memory and feeds the GPU one batch at a time. That is usually much more efficient than trying to move every window onto the GPU all at once.


In [ ]:
if not CNN_RUN:
    print("CNN model setup skipped.")
else:
    # Fix PyTorch randomness so reruns are easier to compare.
    torch.manual_seed(SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type == "cuda":
        torch.backends.cudnn.benchmark = True

    # TinyQCNet is a compact 1D CNN: it scans across time and learns short temporal patterns.
    # Input tensors have shape (batch, sensor_channels, time_steps).
    class TinyQCNet(nn.Module):
        def __init__(self, channels: int, output_dim: int) -> None:
            super().__init__()
            self.output_dim = output_dim
            # The feature extractor applies convolution filters over neighbouring timestamps.
            self.features = nn.Sequential(
                nn.Conv1d(channels, CNN_CONFIG["conv_channels"][0], kernel_size=7, padding=3),
                nn.ReLU(),
                nn.Conv1d(CNN_CONFIG["conv_channels"][0], CNN_CONFIG["conv_channels"][1], kernel_size=5, padding=2),
                nn.ReLU(),
                nn.Dropout(CNN_CONFIG["dropout"]),
            )
            if CNN_OUTPUT_MODE == "window":
                # Pool across time when the model should emit one label for the whole window.
                self.pool = nn.AdaptiveAvgPool1d(1)
                self.head = nn.Linear(CNN_CONFIG["conv_channels"][1], output_dim)
            else:
                # A 1x1 convolution keeps the time axis and emits one prediction per timestamp.
                self.head = nn.Conv1d(CNN_CONFIG["conv_channels"][1], output_dim, kernel_size=1)

        def forward(self, x: torch.Tensor) -> torch.Tensor:
            # First transform the raw sensor channels into learned temporal features.
            x = self.features(x)
            if CNN_OUTPUT_MODE == "window":
                x = self.pool(x).squeeze(-1)
                return self.head(x)
            logits = self.head(x)
            if self.output_dim == 1:
                return logits.squeeze(1)
            return logits.transpose(1, 2)

    # Choose a loss function that matches the target type.
    # Multiclass uses one integer class per example; binary uses a single logit and class weighting.
    if task_mode == "multiclass":
        train_targets_tensor = torch.from_numpy(y_train).long()
        valid_targets_tensor = torch.from_numpy(y_valid).long()
        test_targets_tensor = torch.from_numpy(y_test).long()
        # Weight rare classes more heavily so the model does not learn to ignore issue labels.
        class_counts = np.bincount(y_train.reshape(-1), minlength=len(class_labels)).clip(min=1)
        class_weights = y_train.size / (len(class_counts) * class_counts)
        loss_fn = nn.CrossEntropyLoss(weight=torch.tensor(class_weights, dtype=torch.float32, device=device))
        output_dim = len(class_labels)
    else:
        train_targets_tensor = torch.from_numpy(y_train.astype(np.float32))
        valid_targets_tensor = torch.from_numpy(y_valid.astype(np.float32))
        test_targets_tensor = torch.from_numpy(y_test.astype(np.float32))
        # pos_weight gives the issue class more influence when issues are rare.
        positive_count = max(float(y_train.sum()), 1.0)
        negative_count = max(float(y_train.size - y_train.sum()), 1.0)
        pos_weight = torch.tensor([negative_count / positive_count], dtype=torch.float32, device=device)
        loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        output_dim = 1

    # TensorDataset pairs each input window with its target label for DataLoader batching.
    train_dataset = TensorDataset(torch.from_numpy(np.ascontiguousarray(X_train)), train_targets_tensor)
    valid_dataset = TensorDataset(torch.from_numpy(np.ascontiguousarray(X_valid)), valid_targets_tensor)
    test_dataset = TensorDataset(torch.from_numpy(np.ascontiguousarray(X_test)), test_targets_tensor)

    # DataLoader options are collected once so train/validation/test loaders stay consistent.
    loader_kwargs = {}
    num_workers = int(CNN_LOADER_CONFIG["num_workers"])
    if num_workers > 0:
        loader_kwargs["num_workers"] = num_workers
        loader_kwargs["persistent_workers"] = bool(CNN_LOADER_CONFIG["persistent_workers"])
        loader_kwargs["prefetch_factor"] = int(CNN_LOADER_CONFIG["prefetch_factor"])
    if CNN_LOADER_CONFIG["pin_memory"] and device.type == "cuda":
        loader_kwargs["pin_memory"] = True
    use_non_blocking = bool(loader_kwargs.get("pin_memory", False) and device.type == "cuda")

    # The notebook no longer needs the large NumPy arrays after TensorDatasets are built.
    # Deleting them reduces memory pressure before training starts.
    del X_train, X_valid, X_test, y_train, y_valid, y_test
    del train_targets_tensor, valid_targets_tensor, test_targets_tensor
    import gc
    gc.collect()

    train_loader = DataLoader(
        train_dataset,
        batch_size=CNN_CONFIG["batch_size"],
        shuffle=True,
        **loader_kwargs,
    )
    valid_loader = DataLoader(
        valid_dataset,
        batch_size=CNN_CONFIG["batch_size"],
        shuffle=False,
        **loader_kwargs,
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=CNN_CONFIG["batch_size"],
        shuffle=False,
        **loader_kwargs,
    )

    # Create the model and optimiser. AdamW is a good default for neural networks with weight decay.
    model = TinyQCNet(channels=len(measurement_columns), output_dim=output_dim).to(device)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=CNN_CONFIG["learning_rate"],
        weight_decay=CNN_CONFIG["weight_decay"],
    )
    # Reduce the learning rate when validation F1 stops improving.
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=CNN_CONFIG["lr_decay_factor"],
        patience=CNN_CONFIG["lr_patience"],
    )

    # These variables track the best validation checkpoint and support early stopping.
    best_metric = -np.inf
    best_epoch = 0
    patience_counter = 0
    history = []
    best_state = None

    # One shared epoch function handles both training and evaluation.
    # training=True enables gradients and optimiser steps; training=False only scores batches.
    def run_epoch(loader, training: bool) -> tuple[float, np.ndarray, np.ndarray]:
        model.train(training)
        total_loss = 0.0
        predictions = []
        targets = []
        for batch_x, batch_y in loader:
            batch_x = batch_x.to(device, non_blocking=use_non_blocking)
            batch_y = batch_y.to(device, non_blocking=use_non_blocking)
            with torch.set_grad_enabled(training):
                logits = model(batch_x)
                # The logits shape differs for window-level vs per-timestep predictions,
                # so multiclass loss receives the class dimension in the place PyTorch expects.
                if task_mode == "multiclass":
                    if CNN_OUTPUT_MODE == "window":
                        loss = loss_fn(logits, batch_y)
                        batch_predictions = logits.argmax(dim=1)
                    else:
                        loss = loss_fn(logits.permute(0, 2, 1), batch_y)
                        batch_predictions = logits.argmax(dim=-1)
                else:
                    loss = loss_fn(logits, batch_y)
                    batch_predictions = (torch.sigmoid(logits) >= 0.5).long()
                if training:
                    # Clear old gradients, backpropagate this batch, optionally clip, then update weights.
                    optimizer.zero_grad(set_to_none=True)
                    loss.backward()
                    if CNN_CONFIG["gradient_clip_norm"]:
                        torch.nn.utils.clip_grad_norm_(model.parameters(), CNN_CONFIG["gradient_clip_norm"])
                    optimizer.step()
            total_loss += float(loss.item()) * len(batch_x)
            # Store flattened predictions/targets so F1 can be computed across the whole split.
            predictions.append(batch_predictions.detach().cpu().reshape(-1).numpy())
            targets.append(batch_y.detach().cpu().reshape(-1).numpy())
        predictions_array = np.concatenate(predictions) if predictions else np.array([])
        targets_array = np.concatenate(targets) if targets else np.array([])
        average_loss = total_loss / max(len(loader.dataset), 1)
        return average_loss, predictions_array, targets_array

    print(
        {
            "device": str(device),
            "output_mode": CNN_OUTPUT_MODE,
            "train_batches": len(train_loader),
            "validation_batches": len(valid_loader),
            "test_batches": len(test_loader),
            "loader_config": CNN_LOADER_CONFIG,
        }
    )


### CNN Step 3: Train with Validation Monitoring

This cell runs the optimisation loop.

We are following standard training practice:

- train on the training split,
- score the model on the validation split after each epoch,
- save the best checkpoint seen so far,
- stop early if the validation metric stops improving.

Watch the printed validation F1 as the run progresses. When `output_mode="window"`, that F1 is computed per window. When `output_mode="per_timestep"`, it is computed across all timestamps in the validation windows.


In [ ]:
if not CNN_RUN:
    print("CNN fit skipped.")
else:
    # Train for a small number of epochs and evaluate validation performance after each pass.
    for epoch in range(1, CNN_CONFIG["epochs"] + 1):
        # The training pass updates model weights; predictions are not needed here.
        train_loss, _, _ = run_epoch(train_loader, training=True)
        # The validation pass does not update weights; it estimates generalisation during training.
        valid_loss, valid_preds, valid_targets = run_epoch(valid_loader, training=False)
        valid_f1 = f1_score(valid_targets, valid_preds, average=report_average(task_mode), zero_division=0)
        # Validation F1 drives the learning-rate scheduler and the best-checkpoint decision.
        scheduler.step(valid_f1)
        history.append(
            {
                "epoch": epoch,
                "train_loss": train_loss,
                "valid_loss": valid_loss,
                "valid_f1": valid_f1,
                "learning_rate": optimizer.param_groups[0]["lr"],
            }
        )
        print(
            f"Epoch {epoch:>2} | train_loss={train_loss:.4f} | valid_loss={valid_loss:.4f} | valid_f1={valid_f1:.4f}"
        )

        # Save a checkpoint whenever validation F1 improves by at least min_delta.
        if valid_f1 > best_metric + CNN_CONFIG["min_delta"]:
            best_metric = valid_f1
            best_epoch = epoch
            patience_counter = 0
            best_state = copy.deepcopy(model.state_dict())
            torch.save(
                {
                    "model_state_dict": best_state,
                    "task_mode": task_mode,
                    "class_labels": class_labels,
                    "feature_columns": measurement_columns,
                    "window_size": WINDOW_SIZE,
                    "output_mode": CNN_OUTPUT_MODE,
                    "config": {**CNN_CONFIG, **CNN_LOADER_CONFIG},
                },
                CNN_MODEL_PATH,
            )
        else:
            # Early stopping protects the notebook from spending epochs on a stalled model.
            patience_counter += 1
            if patience_counter >= CNN_CONFIG["patience"]:
                print(f"Early stopping triggered at epoch {epoch}")
                break


### CNN Step 4: Reload the Best Checkpoint and Evaluate

Training loss alone is not enough. In this final step we:

- reload the weights that gave the best validation F1,
- run the held-out test split once,
- inspect the classification report and confusion matrix.

This keeps the evaluation honest and makes it clear that the validation set guided model selection, while the test set is reserved for final reporting.


In [ ]:
if not CNN_RUN:
    print("CNN evaluation skipped.")
else:
    # Reload the best validation checkpoint before touching the test split.
    if best_state is None:
        raise RuntimeError("CNN training did not produce a valid checkpoint")

    model.load_state_dict(best_state)
    # Test results are reported once, after model selection is finished.
    test_loss, test_preds, test_targets = run_epoch(test_loader, training=False)

    print(pd.DataFrame(history).tail())
    print(
        {
            "output_mode": CNN_CONFIG["output_mode"],
            "windows": len(train_dataset) + len(valid_dataset) + len(test_dataset),
            "device": str(device),
            "best_validation_f1": float(best_metric),
            "best_epoch": best_epoch,
            "test_loss": float(test_loss),
            "saved_model": str(CNN_MODEL_PATH),
            "loader_config": CNN_LOADER_CONFIG,
        }
    )

    # Build human-readable report labels for either multiclass or binary mode.
    if task_mode == "multiclass":
        report_labels = list(range(len(class_labels)))
        report_names = [str(label) for label in class_labels]
    else:
        report_labels = [0, 1]
        report_names = ["0", "1"]

    print(
        classification_report(
            test_targets,
            test_preds,
            labels=report_labels,
            target_names=report_names,
            zero_division=0,
        )
    )

    # Plot learning curves and a normalised confusion matrix to show both training behaviour and errors.
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    history_frame = pd.DataFrame(history)
    axes[0].plot(history_frame["epoch"], history_frame["train_loss"], label="train")
    axes[0].plot(history_frame["epoch"], history_frame["valid_loss"], label="validation")
    axes[0].set_title("CNN loss history")
    axes[0].legend()
    ConfusionMatrixDisplay.from_predictions(
        test_targets,
        test_preds,
        labels=report_labels,
        display_labels=report_names,
        normalize="true",
        xticks_rotation=45,
        ax=axes[1],
    )
    axes[1].set_title("CNN test confusion matrix")
    plt.tight_layout()
    plt.show()


### Date-Range Demo: See CNN Predictions on a Specific Interval

This demo follows the same `output_mode` toggle as the training section:

- `"window"` shows one true/predicted label span per window,
- `"per_timestep"` shows one true/predicted label span per timestamp.


In [ ]:
# Leave start/end as None to auto-select an interesting test interval.
# Set explicit timestamps here when you want to inspect a specific deployment period.
CNN_RANGE_START = None
CNN_RANGE_END = None
CNN_AUTO_SELECT_TEST_RANGE = True
CNN_MAX_POINTS_TO_PLOT = 1024

print(
    {
        "CNN_RANGE_START": CNN_RANGE_START,
        "CNN_RANGE_END": CNN_RANGE_END,
        "CNN_AUTO_SELECT_TEST_RANGE": CNN_AUTO_SELECT_TEST_RANGE,
        "CNN_MAX_POINTS_TO_PLOT": CNN_MAX_POINTS_TO_PLOT,
    }
)


In [ ]:
cnn_interval_metrics = None

# Keep this guard because `model`, `device`, `channel_mean`, and `channel_std`
# are created only when the CNN training cells have run successfully.
if not CNN_RUN:
    cnn_demo_result = show_cnn_interval_demo(cnn_run=False)
else:
    cnn_demo_result = show_cnn_interval_demo(
        cnn_run=CNN_RUN,
        # CNN_CONFIG controls the prediction shape and runtime:
        # - output_mode="window" gives one label per window;
        #   output_mode="per_timestep" gives one label per timestamp.
        # - window_size changes how much history each CNN example sees.
        # - batch_size changes how many examples are predicted at once.
        cnn_config=CNN_CONFIG,
        # These objects come from the CNN training cells above.
        model=model,
        device=device,
        channel_mean=channel_mean,
        channel_std=channel_std,
        # The helper auto-selects from `test_df` by default so this visual check
        # usually stays on held-out data. Set CNN_RANGE_START/CNN_RANGE_END in
        # the previous cell when you want to inspect a specific time period.
        train_df=train_df,
        valid_df=valid_df,
        test_df=test_df,
        range_start=CNN_RANGE_START,
        range_end=CNN_RANGE_END,
        auto_select_test_range=CNN_AUTO_SELECT_TEST_RANGE,
        max_points_to_plot=CNN_MAX_POINTS_TO_PLOT,
        # These inputs tell the helper how to reload the selected raw rows and
        # rebuild the same model-ready columns used during training.
        metadata=metadata,
        row_cache_dir=ROW_CACHE_DIR,
        row_use_columns=ROW_USE_COLUMNS,
        target_flag=TARGET_FLAG,
        measurement_columns=measurement_columns,
        task_mode=task_mode,
        class_labels=class_labels,
        # Plot labels should match the active dataset. For conductivity-plug
        # runs this means custom `ml_label` meanings rather than generic QC flags.
        plot_measurement_column=PLOT_MEASUREMENT_COLUMN,
        plot_secondary_column=PLOT_SECONDARY_COLUMN,
        flag_display_name=FLAG_EXAMPLE_DISPLAY_NAME,
        label_meanings=FLAG_EXAMPLE_LABEL_MEANINGS,
        flag_palette=DEFAULT_FLAG_PALETTE,
    )
    cnn_interval_metrics = cnn_demo_result["interval_metrics"]


### Reflection Questions: Baseline CNN

1. Which output mode is active here: one label per window or one label per timestamp? How does that change what each mistake means?
2. If `output_mode="window"`, does compressing a full window into one label hide short events? If `output_mode="per_timestep"`, are the predicted boundaries sharp enough to be useful?
3. Would you improve this result first by changing `window_size`, the window label rule, the input features, or the CNN architecture/training settings?

Add your own notes or follow-up questions here:

- 
- 
- 


## Part 7 — Transformer

The transformer keeps the same supervised task but changes how the model uses context inside each sequence window.


#### Skip Ahead To This ML Section

Use the next cell if you want to jump straight to this ML section, or if you already ran earlier sections but your kernel state was cleared. It restores the cache paths, dataset-specific label names/meanings, reviewed modelling rows, train/validation/test split, and train-only subset.

Uncomment the cell, adjust only the inputs you care about, and run it before continuing.


In [ ]:
# # Uncomment and run this cell to jump straight to this ML section.
# from pathlib import Path
# import sys
#
# NOTEBOOK_ROOT = Path.cwd()
# for candidate_root in [NOTEBOOK_ROOT, *NOTEBOOK_ROOT.parents]:
#     if (candidate_root / "notebooks").exists() and (candidate_root / "scripts").exists():
#         NOTEBOOK_ROOT = candidate_root
#         break
# if str(NOTEBOOK_ROOT) not in sys.path:
#     sys.path.insert(0, str(NOTEBOOK_ROOT))
#
# from scripts.session1_resume_utils import load_ml_section_state
#
# globals().update(
#     load_ml_section_state(
#         notebook_root=NOTEBOOK_ROOT,  # Repo root; usually leave this alone.
#         dataset_profile_id="ctd_conductivity",  # Dataset preset: "ctd_conductivity", "fluorometer_turbidity", "oxygen", or "conductivity_plugs".
#         data_fraction=0.1,  # Dataset size: 0.1 is a quick 10% run, 0.9 is a larger 90% run, and 1.0 removes row caps.
#         split_strategy="global_contiguous",  # Split style: "global_contiguous", "per_source_contiguous", or "interleaved_blocks".
#         train_subset_strategy="time_spread",  # Train-only subset: "time_spread", "issue_focused", or "balanced_reviewed".
#     )
# )


### Sequential Model: Small Transformer Encoder

If you have used ChatGPT or seen a text-to-image model, you have already interacted with a **transformer**.

Transformers were introduced in 2017 in *Attention Is All You Need*, and they quickly became the dominant architecture for sequence modelling. They are most famous in language, but the underlying math is not tied to text. The same ideas are useful for scalar sensor streams, audio, and video.

A simple reason they matter is that they replaced the older "read one step, then the next step" style used by RNNs and LSTMs.

- **RNN/LSTM:** process sequences step by step, which makes training slower and can make long-range context fade.
- **Transformer:** processes the full window together and learns which positions should pay attention to which other positions.

That core operation is called **self-attention**. In plain language, each timestep asks: *which other timesteps in this window are most relevant to understanding me right now?*

For ONC-style data, that is appealing whenever a local measurement only makes sense in a wider context, such as linking a short anomaly to something that happened much earlier or later in the same window.

In this notebook we use a small **encoder-only transformer** for classification. We are not generating text, so we only keep the encoder side and attach a classifier on top.

If you want a fuller explanation after this section, here are the most useful references:

- [ONC Confluence: Transformers overview](https://internal.oceannetworks.ca/x/mwDcDg)
- [3blue1brown: Attention in transformers, step-by-step](https://www.youtube.com/watch?v=eMlx5fFNoYc)
- [The Illustrated Transformer](https://jalammar.github.io/illustrated-transformer/)
- [Attention Is All You Need](https://arxiv.org/abs/1706.03762)


![Transformer model architecture](../assets/session1/transformer_model_architecture.png)

This is the big-picture map. The original transformer has:

- an **encoder stack** on the left,
- a **decoder stack** on the right.

In this notebook we only use the **encoder stack** and attach a classifier on top. So when you look at the rest of the section, focus mostly on the left half of this picture rather than trying to memorize the whole diagram.

Image credit: Arz, Wikimedia Commons, CC BY-SA 4.0. Source: [File:The-Transformer-model-architecture.png](https://commons.wikimedia.org/wiki/File:The-Transformer-model-architecture.png)


![Detailed encoder self-attention diagram](../assets/session1/transformer_encoder_self_attention_detailed.png)

This picture zooms in on the core idea of **self-attention**.

You do not need to memorize every symbol here. The important story is:

- each position creates a few learned views of itself,
- those views are compared against other positions,
- the model turns those comparisons into attention weights,
- then it blends information from the whole window into a new representation.

In plain language: a timestep can decide which other timesteps are worth listening to.

If you want the clearest visual explanation of this idea, the 3blue1brown video linked above is the best companion resource for this section.

Image credit: dvgodoy, Wikimedia Commons, CC BY-SA 4.0. Source: [File:Encoder self-attention, detailed diagram.png](https://commons.wikimedia.org/wiki/File:Encoder_self-attention,_detailed_diagram.png)


![Positional encoding heatmap](../assets/session1/transformer_positional_encoding.png)

One natural question is: if attention can compare all positions to all other positions, how does the model know **where** each timestep sits in the sequence?

That is the job of **positional encoding**.

This heatmap shows that each position gets its own pattern added to the input representation. That gives the transformer access to order information, so it can tell the difference between "this happened early in the window" and "this happened later".

Image credit: Mdbartosz, Wikimedia Commons, CC BY-SA 4.0. Source: [File:Positional encoding.png](https://commons.wikimedia.org/wiki/File:Positional_encoding.png)


### Transformer Settings

Main model variables:

- `window_size`: number of time steps in each window.
- `output_mode`: choose `"window"` for one label per window or `"per_timestep"` for one label at every time step.
- `d_model`: internal embedding size for each position.
- `nhead`: number of attention heads. This must divide `d_model`.
- `num_layers`: number of stacked encoder layers.
- `dim_feedforward`: hidden size of the feed-forward sublayer inside each encoder block.
- `dropout`: dropout used inside the model.
- `pooling`: how we reduce the sequence to one vector for classification. This only matters when `output_mode="window"`.
- `label_reduction`: how row-level labels become one window label. This only matters when `output_mode="window"`.

Training variables:

- `epochs`, `batch_size`, `learning_rate`, `weight_decay`
- `patience`, `min_delta`
- `lr_decay_factor`, `lr_patience`
- `gradient_clip_norm`

Practical note:

- larger `window_size`, `d_model`, or `num_layers` can improve context capacity,
- but they also increase memory use and training time.
- `num_workers > 0` can also increase memory use a lot on Jupyter kernels, because worker
  processes may copy parts of the parent process. Keep it at `0` unless you know the
  environment has headroom.
- `pin_memory` mainly helps when you are actually training on a CUDA device.

Dataset-aware default:

- the turbidity profile starts in `"per_timestep"` mode because the most interesting events are short and windows are often mixed,
- the other profiles start in `"window"` mode so you can begin with one simpler sequence-classification baseline.

A simple mental model for the main knobs:

- `window_size` controls how much time context the model can see,
- `d_model` controls how much representational space each timestep gets,
- `nhead` controls how many different attention patterns the model can learn in parallel,
- `num_layers` controls how many times the sequence is reprocessed with attention.


In [ ]:
TRANSFORMER_CONFIG = {
    "window_size": 128,
    "output_mode": DEFAULT_SEQUENCE_OUTPUT_MODE,
    "d_model": 64,
    "nhead": 4,
    "num_layers": 2,
    "dim_feedforward": 128,
    "dropout": 0.2,
    "pooling": "mean",
    "label_reduction": "worst",
    "epochs": 8,
    "batch_size": 128,
    "learning_rate": 1e-3,
    "weight_decay": 1e-4,
    "patience": 3,
    "min_delta": 0.001,
    "lr_decay_factor": 0.5,
    "lr_patience": 1,
    "gradient_clip_norm": 1.0,
}

_loader_torch = globals().get("torch", None)
TRANSFORMER_LOADER_CONFIG = {
    "num_workers": 0,
    "pin_memory": bool(_loader_torch is not None and _loader_torch.cuda.is_available()),
    "persistent_workers": False,
    "prefetch_factor": 2,
}

display(pd.Series(TRANSFORMER_CONFIG, name="value").rename_axis("transformer_parameter").to_frame())
display(pd.Series(TRANSFORMER_LOADER_CONFIG, name="value").rename_axis("loader_parameter").to_frame())


### Transformer Step 1: Prepare Windowed Sequence Data

The transformer uses the same fixed split as the CNN, but it keeps the sequence in its original time-major layout.

In other words:

- CNN sees `[batch, channels, time]`,
- transformer sees `[batch, time, features]`.

Just like the CNN, the transformer here uses windows of the selected measurement columns rather than the full Random Forest feature table. So it gets temporal context directly from the sequence, not from handcrafted calendar or delta features.

The `output_mode` toggle also matters here:

- `"window"` trains one prediction for the whole window,
- `"per_timestep"` trains one prediction at every position in the window.

That makes this step a nice comparison point between two sequence models trained on the same scalar QC problem.


### Transformer Step 0: Release Neural Model Memory And Load PyTorch

If you ran the CNN section just before this one, the notebook may still be holding large tensors, `DataLoader` objects, and model state.

This cleanup cell frees those objects before the transformer starts. It also imports PyTorch here so the transformer section can be run even if you skipped the CNN cells.


In [ ]:
import gc

try:
    import torch
    from torch import nn
    from torch.utils.data import DataLoader, TensorDataset

    PYTORCH_READY = True
except ImportError as exc:
    PYTORCH_READY = False
    print("PyTorch is not installed in this environment.")
    raise SystemExit(exc)

# Clear large sequence-model objects left behind by earlier sections before
# building transformer windows. This is especially important on shared
# Jupyter kernels where CPU RAM is limited.
released_names = []
for name in [
    "cnn_bundle",
    "sequence_split_frames_materialized",
    "train_dataset",
    "valid_dataset",
    "test_dataset",
    "train_loader",
    "valid_loader",
    "test_loader",
    "train_targets_tensor",
    "valid_targets_tensor",
    "test_targets_tensor",
    "X_train",
    "X_valid",
    "X_test",
    "y_train",
    "y_valid",
    "y_test",
    "model",
    "optimizer",
    "scheduler",
    "loss_fn",
    "best_state",
    "history",
]:
    if name in globals():
        del globals()[name]
        released_names.append(name)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(
    {
        "released_objects": released_names,
        "released_count": len(released_names),
        "torch": torch.__version__,
        "cuda_available": torch.cuda.is_available(),
    }
)


In [ ]:
if not globals().get("PYTORCH_READY", False):
    TRANSFORMER_RUN = False
    print("Transformer training cell skipped because PyTorch is not available in the current notebook state.")
else:
    sequence_split_frames_materialized = materialize_reviewed_split_frames(
        selected_paths,
        {"train": train_df, "validation": valid_df, "test": test_df},
        columns=ROW_USE_COLUMNS,
        target_flag=TARGET_FLAG,
        task_mode=task_mode,
        good_labels=GOOD_LABELS,
        issue_labels=ISSUE_LABELS,
    )
    transformer_window_size = TRANSFORMER_CONFIG["window_size"]
    TRANSFORMER_OUTPUT_MODE = TRANSFORMER_CONFIG["output_mode"]
    transformer_bundle = build_sequence_split_bundle_from_frames(
        sequence_split_frames_materialized,
        measurement_columns=measurement_columns,
        target_column="model_target",
        task_mode=task_mode,
        output_mode=TRANSFORMER_OUTPUT_MODE,
        window_size=transformer_window_size,
        label_reduction=TRANSFORMER_CONFIG["label_reduction"],
    )
    transformer_class_labels = transformer_bundle.class_labels

    if task_mode == "multiclass" and len(transformer_class_labels) < 2:
        TRANSFORMER_RUN = False
        print(
            {
                "output_mode": TRANSFORMER_OUTPUT_MODE,
                "windows_total": int(len(transformer_bundle.X_train) + len(transformer_bundle.X_valid) + len(transformer_bundle.X_test)),
                "active_labels": transformer_class_labels,
                "skip_reason": "Need at least two active classes to train the transformer in multiclass mode.",
            }
        )
    elif len(transformer_bundle.X_train) == 0 or len(transformer_bundle.X_valid) == 0 or len(transformer_bundle.X_test) == 0:
        TRANSFORMER_RUN = False
        print(
            {
                "output_mode": TRANSFORMER_OUTPUT_MODE,
                "split_strategy": FIXED_SPLIT_STRATEGY,
                "train_subset_strategy": TRAIN_SUBSET_STRATEGY,
                "skip_reason": "At least one split produced no full windows for this configuration.",
            }
        )
    else:
        X_train = transformer_bundle.X_train
        X_valid = transformer_bundle.X_valid
        X_test = transformer_bundle.X_test
        y_train = transformer_bundle.y_train
        y_valid = transformer_bundle.y_valid
        y_test = transformer_bundle.y_test

        feature_mean = X_train.mean(axis=(0, 1), keepdims=True)
        feature_std = X_train.std(axis=(0, 1), keepdims=True) + 1e-6
        X_train = ((X_train - feature_mean) / feature_std).astype(np.float32, copy=False)
        X_valid = ((X_valid - feature_mean) / feature_std).astype(np.float32, copy=False)
        X_test = ((X_test - feature_mean) / feature_std).astype(np.float32, copy=False)

        del transformer_bundle, sequence_split_frames_materialized
        import gc
        gc.collect()

        TRANSFORMER_RUN = True
        print(
            {
                "output_mode": TRANSFORMER_OUTPUT_MODE,
                "split_strategy": FIXED_SPLIT_STRATEGY,
                "train_subset_strategy": TRAIN_SUBSET_STRATEGY,
                "fixed_split_block_rows": FIXED_SPLIT_BLOCK_ROWS if FIXED_SPLIT_STRATEGY in {"interleaved_blocks", "episode_aware"} else None,
                "windows_total": int(len(X_train) + len(X_valid) + len(X_test)),
                "train_windows": int(len(X_train)),
                "validation_windows": int(len(X_valid)),
                "test_windows": int(len(X_test)),
                "window_size": int(transformer_window_size),
                "features_per_step": int(len(measurement_columns)),
            }
        )


### Transformer Step 2: Build Attention Blocks, Loss, and DataLoaders

The transformer needs a few ingredients that are specific to attention models:

- an input projection that maps raw sensor features into `d_model`,
- a positional embedding so the model knows where each time step sits in the window,
- stacked encoder blocks with multi-head attention,
- and, in window mode, a pooling rule that turns the whole window into one classification vector.

The output toggle changes the last step:

- in `"window"` mode, we pool the encoded sequence and classify once,
- in `"per_timestep"` mode, we classify every encoded position directly.

Everything else should feel familiar from the CNN section: loss, optimiser, scheduler, and `DataLoader` setup.


In [ ]:
if not TRANSFORMER_RUN:
    print("Transformer model setup skipped.")
else:
    torch.manual_seed(SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type == "cuda":
        torch.backends.cudnn.benchmark = True

    class TinyQCTransformer(nn.Module):
        def __init__(self, input_dim: int, output_dim: int) -> None:
            super().__init__()
            self.output_dim = output_dim
            self.input_projection = nn.Linear(input_dim, TRANSFORMER_CONFIG["d_model"])
            self.position_embedding = nn.Parameter(
                torch.zeros(1, TRANSFORMER_CONFIG["window_size"], TRANSFORMER_CONFIG["d_model"])
            )
            encoder_layer = nn.TransformerEncoderLayer(
                d_model=TRANSFORMER_CONFIG["d_model"],
                nhead=TRANSFORMER_CONFIG["nhead"],
                dim_feedforward=TRANSFORMER_CONFIG["dim_feedforward"],
                dropout=TRANSFORMER_CONFIG["dropout"],
                activation="gelu",
                batch_first=True,
            )
            self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=TRANSFORMER_CONFIG["num_layers"])
            self.norm = nn.LayerNorm(TRANSFORMER_CONFIG["d_model"])
            self.head = nn.Linear(TRANSFORMER_CONFIG["d_model"], output_dim)

        def forward(self, x: torch.Tensor) -> torch.Tensor:
            x = self.input_projection(x)
            x = x + self.position_embedding[:, : x.size(1)]
            x = self.encoder(x)
            if TRANSFORMER_OUTPUT_MODE == "window":
                if TRANSFORMER_CONFIG["pooling"] == "last":
                    pooled = x[:, -1, :]
                else:
                    pooled = x.mean(dim=1)
                pooled = self.norm(pooled)
                return self.head(pooled)
            x = self.norm(x)
            logits = self.head(x)
            if self.output_dim == 1:
                return logits.squeeze(-1)
            return logits

    if task_mode == "multiclass":
        train_targets_tensor = torch.from_numpy(y_train).long()
        valid_targets_tensor = torch.from_numpy(y_valid).long()
        test_targets_tensor = torch.from_numpy(y_test).long()
        class_counts = np.bincount(y_train.reshape(-1), minlength=len(transformer_class_labels)).clip(min=1)
        class_weights = y_train.size / (len(class_counts) * class_counts)
        loss_fn = nn.CrossEntropyLoss(weight=torch.tensor(class_weights, dtype=torch.float32, device=device))
        output_dim = len(transformer_class_labels)
    else:
        train_targets_tensor = torch.from_numpy(y_train.astype(np.float32))
        valid_targets_tensor = torch.from_numpy(y_valid.astype(np.float32))
        test_targets_tensor = torch.from_numpy(y_test.astype(np.float32))
        positive_count = max(float(y_train.sum()), 1.0)
        negative_count = max(float(y_train.size - y_train.sum()), 1.0)
        pos_weight = torch.tensor([negative_count / positive_count], dtype=torch.float32, device=device)
        loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        output_dim = 1

    train_dataset = TensorDataset(torch.from_numpy(np.ascontiguousarray(X_train)), train_targets_tensor)
    valid_dataset = TensorDataset(torch.from_numpy(np.ascontiguousarray(X_valid)), valid_targets_tensor)
    test_dataset = TensorDataset(torch.from_numpy(np.ascontiguousarray(X_test)), test_targets_tensor)

    loader_kwargs = {}
    num_workers = int(TRANSFORMER_LOADER_CONFIG["num_workers"])
    if num_workers > 0:
        loader_kwargs["num_workers"] = num_workers
        loader_kwargs["persistent_workers"] = bool(TRANSFORMER_LOADER_CONFIG["persistent_workers"])
        loader_kwargs["prefetch_factor"] = int(TRANSFORMER_LOADER_CONFIG["prefetch_factor"])
    if TRANSFORMER_LOADER_CONFIG["pin_memory"] and device.type == "cuda":
        loader_kwargs["pin_memory"] = True
    use_non_blocking = bool(loader_kwargs.get("pin_memory", False) and device.type == "cuda")

    del X_train, X_valid, X_test, y_train, y_valid, y_test
    del train_targets_tensor, valid_targets_tensor, test_targets_tensor
    import gc
    gc.collect()

    train_loader = DataLoader(train_dataset, batch_size=TRANSFORMER_CONFIG["batch_size"], shuffle=True, **loader_kwargs)
    valid_loader = DataLoader(valid_dataset, batch_size=TRANSFORMER_CONFIG["batch_size"], shuffle=False, **loader_kwargs)
    test_loader = DataLoader(test_dataset, batch_size=TRANSFORMER_CONFIG["batch_size"], shuffle=False, **loader_kwargs)

    model = TinyQCTransformer(input_dim=len(measurement_columns), output_dim=output_dim).to(device)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=TRANSFORMER_CONFIG["learning_rate"],
        weight_decay=TRANSFORMER_CONFIG["weight_decay"],
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=TRANSFORMER_CONFIG["lr_decay_factor"],
        patience=TRANSFORMER_CONFIG["lr_patience"],
    )

    best_metric = -np.inf
    best_epoch = 0
    patience_counter = 0
    history = []
    best_state = None

    def run_epoch(loader, training: bool) -> tuple[float, np.ndarray, np.ndarray]:
        model.train(training)
        total_loss = 0.0
        predictions = []
        targets = []
        for batch_x, batch_y in loader:
            batch_x = batch_x.to(device, non_blocking=use_non_blocking)
            batch_y = batch_y.to(device, non_blocking=use_non_blocking)
            with torch.set_grad_enabled(training):
                logits = model(batch_x)
                if task_mode == "multiclass":
                    if TRANSFORMER_OUTPUT_MODE == "window":
                        loss = loss_fn(logits, batch_y)
                        batch_predictions = logits.argmax(dim=1)
                    else:
                        loss = loss_fn(logits.permute(0, 2, 1), batch_y)
                        batch_predictions = logits.argmax(dim=-1)
                else:
                    loss = loss_fn(logits, batch_y)
                    batch_predictions = (torch.sigmoid(logits) >= 0.5).long()
                if training:
                    optimizer.zero_grad(set_to_none=True)
                    loss.backward()
                    if TRANSFORMER_CONFIG["gradient_clip_norm"]:
                        torch.nn.utils.clip_grad_norm_(model.parameters(), TRANSFORMER_CONFIG["gradient_clip_norm"])
                    optimizer.step()
            total_loss += float(loss.item()) * len(batch_x)
            predictions.append(batch_predictions.detach().cpu().reshape(-1).numpy())
            targets.append(batch_y.detach().cpu().reshape(-1).numpy())
        predictions_array = np.concatenate(predictions) if predictions else np.array([])
        targets_array = np.concatenate(targets) if targets else np.array([])
        average_loss = total_loss / max(len(loader.dataset), 1)
        return average_loss, predictions_array, targets_array

    print(
        {
            "device": str(device),
            "output_mode": TRANSFORMER_OUTPUT_MODE,
            "train_batches": len(train_loader),
            "validation_batches": len(valid_loader),
            "test_batches": len(test_loader),
            "loader_config": TRANSFORMER_LOADER_CONFIG,
        }
    )


### Transformer Step 3: Train with Attention-Based Sequence Modelling

The training loop is almost the same as the CNN loop, which is useful pedagogically: once the data pipeline and validation workflow are clear, we can swap in a different model family without changing the whole experimental process.

Just like the CNN section, the reported validation F1 is per window in `"window"` mode and per timestamp in `"per_timestep"` mode.


In [ ]:
if not TRANSFORMER_RUN:
    print("Transformer fit skipped.")
else:
    for epoch in range(1, TRANSFORMER_CONFIG["epochs"] + 1):
        train_loss, _, _ = run_epoch(train_loader, training=True)
        valid_loss, valid_preds, valid_targets = run_epoch(valid_loader, training=False)
        valid_f1 = f1_score(valid_targets, valid_preds, average=report_average(task_mode), zero_division=0)
        scheduler.step(valid_f1)
        history.append(
            {
                "epoch": epoch,
                "train_loss": train_loss,
                "valid_loss": valid_loss,
                "valid_f1": valid_f1,
                "learning_rate": optimizer.param_groups[0]["lr"],
            }
        )
        print(
            f"Epoch {epoch:>2} | train_loss={train_loss:.4f} | valid_loss={valid_loss:.4f} | valid_f1={valid_f1:.4f}"
        )

        if valid_f1 > best_metric + TRANSFORMER_CONFIG["min_delta"]:
            best_metric = valid_f1
            best_epoch = epoch
            patience_counter = 0
            best_state = copy.deepcopy(model.state_dict())
            torch.save(
                {
                    "model_state_dict": best_state,
                    "task_mode": task_mode,
                    "class_labels": transformer_class_labels,
                    "feature_columns": measurement_columns,
                    "window_size": transformer_window_size,
                    "output_mode": TRANSFORMER_OUTPUT_MODE,
                    "config": {**TRANSFORMER_CONFIG, **TRANSFORMER_LOADER_CONFIG},
                },
                TRANSFORMER_MODEL_PATH,
            )
        else:
            patience_counter += 1
            if patience_counter >= TRANSFORMER_CONFIG["patience"]:
                print(f"Early stopping triggered at epoch {epoch}")
                break


### Transformer Step 4: Evaluate the Best Attention Model

Just like the CNN section, we finish by restoring the best validation checkpoint and scoring the held-out test split exactly once.


In [ ]:
if not TRANSFORMER_RUN:
    print("Transformer evaluation skipped.")
else:
    if best_state is None:
        raise RuntimeError("Transformer training did not produce a valid checkpoint")

    model.load_state_dict(best_state)
    test_loss, test_preds, test_targets = run_epoch(test_loader, training=False)

    print(pd.DataFrame(history).tail())
    print(
        {
            "output_mode": TRANSFORMER_CONFIG["output_mode"],
            "windows": len(train_dataset) + len(valid_dataset) + len(test_dataset),
            "device": str(device),
            "best_validation_f1": float(best_metric),
            "best_epoch": best_epoch,
            "test_loss": float(test_loss),
            "saved_model": str(TRANSFORMER_MODEL_PATH),
        }
    )

    if task_mode == "multiclass":
        report_labels = list(range(len(transformer_class_labels)))
        report_names = [str(label) for label in transformer_class_labels]
    else:
        report_labels = [0, 1]
        report_names = ["0", "1"]

    print(
        classification_report(
            test_targets,
            test_preds,
            labels=report_labels,
            target_names=report_names,
            zero_division=0,
        )
    )

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    history_frame = pd.DataFrame(history)
    axes[0].plot(history_frame["epoch"], history_frame["train_loss"], label="train")
    axes[0].plot(history_frame["epoch"], history_frame["valid_loss"], label="validation")
    axes[0].set_title("Transformer loss history")
    axes[0].legend()
    ConfusionMatrixDisplay.from_predictions(
        test_targets,
        test_preds,
        labels=report_labels,
        display_labels=report_names,
        normalize="true",
        xticks_rotation=45,
        ax=axes[1],
    )
    axes[1].set_title("Transformer test confusion matrix")
    plt.tight_layout()
    plt.show()


### Date-Range Demo: See Transformer Predictions on a Specific Interval

This demo follows the same `output_mode` toggle as the training section:

- `"window"` shows one true/predicted label span per window,
- `"per_timestep"` shows one true/predicted label span per timestamp.


In [ ]:
TRANSFORMER_RANGE_START = None
TRANSFORMER_RANGE_END = None
TRANSFORMER_AUTO_SELECT_TEST_RANGE = True
TRANSFORMER_MAX_POINTS_TO_PLOT = 1024

print(
    {
        "TRANSFORMER_RANGE_START": TRANSFORMER_RANGE_START,
        "TRANSFORMER_RANGE_END": TRANSFORMER_RANGE_END,
        "TRANSFORMER_AUTO_SELECT_TEST_RANGE": TRANSFORMER_AUTO_SELECT_TEST_RANGE,
        "TRANSFORMER_MAX_POINTS_TO_PLOT": TRANSFORMER_MAX_POINTS_TO_PLOT,
    }
)


In [ ]:
if not TRANSFORMER_RUN:
    print("Transformer date-range demo skipped because the transformer was not trained in this run.")
else:
    transformer_range_selection = select_time_range(
        test_df,
        time_column="Time UTC",
        label_column=TARGET_FLAG,
        start=TRANSFORMER_RANGE_START,
        end=TRANSFORMER_RANGE_END,
        auto_select=TRANSFORMER_AUTO_SELECT_TEST_RANGE,
        max_points=TRANSFORMER_MAX_POINTS_TO_PLOT,
    )
    transformer_interval_metrics = None

    transformer_interval_rows = load_rows_for_time_range(
        metadata,
        row_cache_dir=Path(ROW_CACHE_DIR),
        start=transformer_range_selection["start"],
        end=transformer_range_selection["end"],
        columns=ROW_USE_COLUMNS,
    )

    if transformer_interval_rows.empty:
        print("No row-level data was found in the requested transformer range.")
    else:
        transformer_interval_model_df, _, _ = build_model_frame(
            transformer_interval_rows,
            target_flag=TARGET_FLAG,
            measurement_columns=measurement_columns,
            task_mode=task_mode,
            model_row_limit=None,
        )
        transformer_interval_model_df = transformer_interval_model_df[
            (transformer_interval_model_df["Time UTC"] >= transformer_range_selection["start"])
            & (transformer_interval_model_df["Time UTC"] <= transformer_range_selection["end"])
        ].reset_index(drop=True)
        transformer_interval_origin = infer_interval_origin(
            transformer_range_selection["start"],
            transformer_range_selection["end"],
            {"train": train_df, "validation": valid_df, "test": test_df},
        )
        transformer_plot_palette = DEFAULT_FLAG_PALETTE if task_mode == "multiclass" else {0: "#1f77b4", 1: "#d62728"}

        if TRANSFORMER_CONFIG["output_mode"] == "window":
            transformer_interval_bundle = build_window_classification_interval_data(
                transformer_interval_model_df,
                feature_columns=measurement_columns,
                target_column="model_target",
                task_mode=task_mode,
                window_size=TRANSFORMER_CONFIG["window_size"],
                label_reduction=TRANSFORMER_CONFIG["label_reduction"],
            )

            if transformer_interval_bundle["window_frame"].empty:
                print("The selected transformer range is shorter than one full window after preprocessing, so the window-level demo is skipped.")
            else:
                transformer_predicted_labels = predict_transformer_window_model(
                    model,
                    transformer_interval_bundle["raw_sequences"],
                    task_mode=task_mode,
                    class_labels=transformer_class_labels,
                    device=str(device),
                    feature_mean=feature_mean,
                    feature_std=feature_std,
                    batch_size=TRANSFORMER_CONFIG["batch_size"],
                )
                transformer_window_frame = transformer_interval_bundle["window_frame"].copy()
                transformer_window_frame["predicted_label"] = transformer_predicted_labels
                transformer_interval_metrics = compute_interval_classification_metrics(
                    transformer_window_frame["true_label"],
                    transformer_window_frame["predicted_label"],
                    labels=transformer_class_labels,
                    average=report_average(task_mode),
                    target_names=[str(label) for label in transformer_class_labels],
                )
                transformer_true_intervals = merge_adjacent_intervals(
                    transformer_window_frame.rename(columns={"window_start": "start", "window_end": "end", "true_label": "label"})[
                        ["start", "end", "label"]
                    ]
                )
                transformer_pred_intervals = merge_adjacent_intervals(
                    transformer_window_frame.rename(columns={"window_start": "start", "window_end": "end", "predicted_label": "label"})[
                        ["start", "end", "label"]
                    ]
                )

                print(
                    {
                        "output_mode": TRANSFORMER_CONFIG["output_mode"],
                        "selection_mode": transformer_range_selection["selection_mode"],
                        "selected_priority_flag": transformer_range_selection["selected_label"],
                        "interval_origin": transformer_interval_origin,
                        "range_start": transformer_range_selection["start"].isoformat(),
                        "range_end": transformer_range_selection["end"].isoformat(),
                        "window_count_in_interval": int(len(transformer_window_frame)),
                        "interval_f1": transformer_interval_metrics["f1"],
                    }
                )
                print(transformer_interval_metrics["report_text"])
                display(
                    pd.DataFrame(
                        {
                            "true_count": transformer_window_frame["true_label"].value_counts().sort_index(),
                            "predicted_count": transformer_window_frame["predicted_label"].value_counts().sort_index(),
                        }
                    ).fillna(0).astype(int)
                )

                transformer_demo_figure = plot_time_series_with_bands(
                    transformer_interval_model_df,
                    band_specs=[
                        {"title": f"True window {FLAG_EXAMPLE_DISPLAY_NAME}", "intervals": transformer_true_intervals, "palette": transformer_plot_palette},
                        {"title": f"Transformer {FLAG_EXAMPLE_DISPLAY_NAME} prediction", "intervals": transformer_pred_intervals, "palette": transformer_plot_palette},
                    ],
                    measurement_column=PLOT_MEASUREMENT_COLUMN,
                    secondary_column=PLOT_SECONDARY_COLUMN,
                    max_points=TRANSFORMER_MAX_POINTS_TO_PLOT,
                    title="Transformer window predictions on a selected time range",
                    label_meanings=FLAG_EXAMPLE_LABEL_MEANINGS,
                    legend_title=FLAG_EXAMPLE_DISPLAY_NAME,
                )
                plt.show()
        else:
            transformer_interval_bundle = build_sequence_label_interval_data(
                transformer_interval_model_df,
                feature_columns=measurement_columns,
                target_column="model_target",
                window_size=TRANSFORMER_CONFIG["window_size"],
            )

            if len(transformer_interval_bundle["raw_sequences"]) == 0:
                print("The selected transformer range is shorter than one full window after preprocessing, so the per-timestep demo is skipped.")
            else:
                transformer_predicted_labels = predict_transformer_sequence_model(
                    model,
                    transformer_interval_bundle["raw_sequences"],
                    task_mode=task_mode,
                    class_labels=transformer_class_labels,
                    device=str(device),
                    feature_mean=feature_mean,
                    feature_std=feature_std,
                    batch_size=TRANSFORMER_CONFIG["batch_size"],
                )
                transformer_point_frame = pd.DataFrame(
                    {
                        "Time UTC": pd.to_datetime(transformer_interval_bundle["raw_times"].reshape(-1), utc=True),
                        "true_label": transformer_interval_bundle["raw_targets"].reshape(-1).astype(int),
                        "predicted_label": transformer_predicted_labels.reshape(-1).astype(int),
                    }
                )
                transformer_interval_metrics = compute_interval_classification_metrics(
                    transformer_point_frame["true_label"],
                    transformer_point_frame["predicted_label"],
                    labels=transformer_class_labels,
                    average=report_average(task_mode),
                    target_names=[str(label) for label in transformer_class_labels],
                )
                transformer_true_intervals = merge_adjacent_intervals(
                    build_labeled_intervals(transformer_point_frame, time_column="Time UTC", label_column="true_label")
                )
                transformer_pred_intervals = merge_adjacent_intervals(
                    build_labeled_intervals(transformer_point_frame, time_column="Time UTC", label_column="predicted_label")
                )

                print(
                    {
                        "output_mode": TRANSFORMER_CONFIG["output_mode"],
                        "selection_mode": transformer_range_selection["selection_mode"],
                        "selected_priority_flag": transformer_range_selection["selected_label"],
                        "interval_origin": transformer_interval_origin,
                        "range_start": transformer_range_selection["start"].isoformat(),
                        "range_end": transformer_range_selection["end"].isoformat(),
                        "point_count_in_interval": int(len(transformer_point_frame)),
                        "interval_f1": transformer_interval_metrics["f1"],
                    }
                )
                print(transformer_interval_metrics["report_text"])
                display(
                    pd.DataFrame(
                        {
                            "true_count": transformer_point_frame["true_label"].value_counts().sort_index(),
                            "predicted_count": transformer_point_frame["predicted_label"].value_counts().sort_index(),
                        }
                    ).fillna(0).astype(int)
                )

                transformer_demo_figure = plot_time_series_with_bands(
                    transformer_interval_model_df,
                    band_specs=[
                        {"title": f"True per-timestep {FLAG_EXAMPLE_DISPLAY_NAME}", "intervals": transformer_true_intervals, "palette": transformer_plot_palette},
                        {"title": f"Transformer per-timestep {FLAG_EXAMPLE_DISPLAY_NAME} prediction", "intervals": transformer_pred_intervals, "palette": transformer_plot_palette},
                    ],
                    measurement_column=PLOT_MEASUREMENT_COLUMN,
                    secondary_column=PLOT_SECONDARY_COLUMN,
                    max_points=TRANSFORMER_MAX_POINTS_TO_PLOT,
                    title="Transformer per-timestep predictions on a selected time range",
                    label_meanings=FLAG_EXAMPLE_LABEL_MEANINGS,
                    legend_title=FLAG_EXAMPLE_DISPLAY_NAME,
                )
                plt.show()

        if transformer_interval_metrics is not None:
            transformer_cm_fig, transformer_cm_ax = plt.subplots(figsize=(5, 4))
            ConfusionMatrixDisplay(
                confusion_matrix=transformer_interval_metrics["confusion_matrix"],
                display_labels=transformer_interval_metrics["display_labels"],
            ).plot(ax=transformer_cm_ax, cmap="Blues", colorbar=False)
            transformer_cm_ax.set_title("Transformer confusion matrix on the selected range")
            plt.tight_layout()
            plt.show()


### Reflection Questions: Transformer

1. Does the selected interval contain longer-context behaviour that you would expect a transformer to use better than the CNN?
2. If the transformer is not clearly helping here, do you think the issue is window length, data volume, or model size?
3. Would you change pooling, positional information, or the target definition first if you wanted a better interval-level result?

Add your own notes or follow-up questions here:

- 
- 
- 


## Part 8 — Reflection and Next Steps

End by comparing what each model family sees, what each one misses, and which changes are most promising for the next round of experiments.


### Wrap-Up, Transformer Note, And Questions To Explore

We now have three very different model families in one notebook:

- a tabular tree ensemble,
- a local-pattern sequence model,
- and a small attention-based sequence model.

That is a useful comparison because each one sees the data a little differently.

Good next questions to explore:

1. Are there other features that might help the Random Forest?
Hint: think about lag features, rolling means, rolling standard deviations, slopes, or relationships between sensors.

2. Are there ways to improve class `3`, `4`, or `9` performance specifically?
Hint: compare the class distribution to the confusion matrix and ask which classes are hardest to distinguish.

3. Does the target itself need to change?
Hint: sometimes a binary `good` vs `issue` target, or a collapsed set of flags, is closer to the operational decision. For conductivity or turbidity, that might mean merging `3` and `4`. For oxygen, it can make more sense to ask whether `1/2` belong together and whether `3/4` belong together, rather than reusing the CTD collapse blindly.

4. What parts of the CNN are worth experimenting with?
Hint: try changing the window length, number of channels, dropout, or the way we reduce row labels into one window label.

5. Could we use even more context from time?
Hint: Random Forest only sees engineered features. CNN sees short windows. Transformer sees relationships across an entire window. There are still other options too, such as TCNs, GRUs/LSTMs, or hierarchical windows.
